# Experiment: Robustness Ablation And Realistic Evaluation

Objective:
- Diagnose whether the current hybrid spectrogram transformer can be made robust in a balanced way instead of only looking strong on easy retrieval cases.
- Upgrade the benchmark from a single-window, centered-query setup into a cache-aware experiment runner with policy-driven augmentation, harder query regimes, multi-window indexing, and FAISS trade-off analysis.

Success criteria:
- The notebook runs top-to-bottom in Google Colab on one GPU runtime.
- It preserves Notebook 2 continuity for the existing baselines.
- It exports final metrics, rankings, plots, failure cases, and one explicit recommendation for Notebook 4.

Non-goals:
- dataset migration
- full architecture rewrites
- multi-GPU training
- UI or deployment work

Expected outputs:
- `final_metrics_long.csv`
- `final_metrics_summary.csv`
- `best_config.json`
- `experiment_registry.json`
- `failure_cases.csv`
- `faiss_sweep_results.csv`
- plot exports
- `notebook3_conclusions.md`


## Setup 
Install only missing notebook dependencies in the active Colab runtime. Deliberately avoid churning the core PyTorch stack unless a package is missing.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

MODULE_TO_PACKAGE: dict[str, str] = {
    "faiss": "faiss-cpu",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "sklearn": "scikit-learn",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}


def install_missing_packages(module_to_package: dict[str, str]) -> list[str]:
    """Install only the packages that are missing from the current kernel."""
    missing_packages = [
        package_name
        for module_name, package_name in module_to_package.items()
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *missing_packages])
    return missing_packages


INSTALLED_NOW = install_missing_packages(MODULE_TO_PACKAGE)
if INSTALLED_NOW:
    print("Installed packages:", INSTALLED_NOW)
else:
    print("All required non-core packages were already present.")


## Setup 
Imports, preliminary runtime inspection, and early failure on unsupported runtimes.

In [ ]:
from __future__ import annotations

import ast
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import shutil
import tempfile
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Literal, TypedDict

import faiss
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from IPython.display import Markdown, display
from sklearn.decomposition import PCA
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, Wav2Vec2FeatureExtractor


def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch for reproducible notebook execution."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def get_gpu_name() -> str:
    """Return the active CUDA device name, or a CPU marker when unavailable."""
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(torch.cuda.current_device())
    return "CPU"


def get_total_vram_gb() -> float:
    """Return total GPU memory in GiB when CUDA is available."""
    if not torch.cuda.is_available():
        return 0.0
    total_bytes = torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory
    return float(total_bytes / (1024 ** 3))


seed_everything(42)

PRELIMINARY_RUNTIME_REPORT = pd.DataFrame(
    [
        {"component": "Python", "value": platform.python_version()},
        {"component": "PyTorch", "value": torch.__version__},
        {"component": "torchaudio", "value": torchaudio.__version__},
        {"component": "Transformers", "value": __import__("transformers").__version__},
        {"component": "FAISS", "value": faiss.__version__},
        {"component": "CUDA available", "value": str(torch.cuda.is_available())},
        {"component": "GPU", "value": get_gpu_name()},
        {"component": "Total VRAM (GiB)", "value": f"{get_total_vram_gb():.2f}"},
        {
            "component": "BF16 supported",
            "value": str(bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())),
        },
    ]
)
display(PRELIMINARY_RUNTIME_REPORT)

if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook is designed for a single-GPU Colab runtime. "
        "CUDA is unavailable, so stop here and switch to a GPU runtime."
    )


## Plan

Hypotheses:
- The hybrid transformer remains the strongest scratch architecture once retrieval becomes harder, but the old `extended` augmentation policy is too blunt.
- Selective augmentation should recover more balanced robustness than the old “stack everything” policy.
- Multi-window indexing should help realistic retrieval enough to justify the extra index size and search complexity.
- Approximate FAISS should remain usable if the quality drop is small relative to the latency and memory savings.

Primary decision metrics:
- `mean_degraded_top1`
- `worst_condition_top1`
- `mrr`
- `latency_ms`
- `index_size_mb`

Execution order:
1. environment bootstrap and Notebook 2 artifact discovery
2. smoke tests on the data/model/index path
3. augmentation-policy training ablations
4. harder retrieval evaluation
5. multi-window indexing and FAISS sweeps
6. error analysis and final recommendation


## Config
Configuration with enough compatibility to reuse the pre-existing Notebook 2 helpers.


In [ ]:
ModelName = Literal["cnn", "hybrid_transformer", "frozen_mert"]
InputMode = Literal["spectrogram", "mert_waveform"]
AugmentationProfileName = str
QueryRegimeName = Literal[
    "clean_current",
    "short_centered",
    "short_offcenter",
    "combined_moderate",
    "multi_segment_same_track",
    "realistic_hard",
]
WindowingSpecName = Literal["single_center", "multi3_even", "multi5_even"]
IndexKind = Literal["exact_ip", "ivfflat", "ivfpq"]
RetrievalConditionName = Literal[
    "clean",
    "noise",
    "pitch",
    "time",
    "lowpass",
    "highpass",
    "combined_moderate",
    "realistic_hard",
]


@dataclass(frozen=True)
class RuntimeInfo:
    """Resolved runtime properties for the active Colab session."""

    python_version: str
    torch_version: str
    torchaudio_version: str
    transformers_version: str
    faiss_version: str
    cuda_available: bool
    device_name: str
    total_vram_gb: float
    bf16_supported: bool


@dataclass(frozen=True)
class PathConfig:
    """Filesystem locations used by the notebook inside Colab."""

    data_root: Path
    output_root: Path
    drive_mount_point: Path
    cache_dir: Path
    checkpoint_dir: Path
    embedding_cache_dir: Path
    index_dir: Path
    metrics_dir: Path
    plot_dir: Path
    export_dir: Path
    log_dir: Path

    @property
    def metadata_zip(self) -> Path:
        return self.data_root / "fma_metadata.zip"

    @property
    def metadata_dir(self) -> Path:
        return self.data_root / "fma_metadata"

    @property
    def audio_zip(self) -> Path:
        return self.data_root / "fma_small.zip"

    @property
    def audio_dir(self) -> Path:
        return self.data_root / "fma_small"

    @property
    def artifact_zip(self) -> Path:
        return self.output_root / "notebook3_robustness_outputs.zip"

    def all_directories(self) -> tuple[Path, ...]:
        return (
            self.data_root,
            self.output_root,
            self.cache_dir,
            self.checkpoint_dir,
            self.embedding_cache_dir,
            self.index_dir,
            self.metrics_dir,
            self.plot_dir,
            self.export_dir,
            self.log_dir,
        )


@dataclass(frozen=True)
class ModelRunConfig:
    """Configuration for a single trainable run in this notebook."""

    model_name: ModelName
    augmentation_profile: AugmentationProfileName
    embed_dim: int
    epochs: int
    learning_rate: float
    weight_decay: float
    temperature: float
    freeze_backbone: bool
    enabled: bool = True

    @property
    def run_id(self) -> str:
        return f"{self.model_name}_{self.augmentation_profile}_embed{self.embed_dim}"


@dataclass(frozen=True)
class ExperimentConfig:
    """Top-level configuration."""

    metadata_url: str
    audio_url: str
    paths: PathConfig
    seed: int
    batch_size: int
    mert_batch_size: int
    num_workers: int
    sample_rate: int
    n_fft: int
    hop_length: int
    n_mels: int
    segment_seconds: float
    amp_enabled: bool
    gradient_checkpointing: bool
    early_stopping_patience: int
    top_k: tuple[int, ...]
    projection_hidden_dim: int
    max_train_tracks: int | None
    max_eval_tracks: int | None
    smoke_test_track_count: int
    gradient_clip_norm: float
    run_presets: tuple[ModelRunConfig, ...]
    strict_artifact_validation: bool
    allow_retrain_missing_baselines: bool
    mount_google_drive: bool
    save_to_drive: bool
    use_torch_compile: bool
    run_training_ablations: bool
    run_faiss_sweep: bool
    run_error_analysis: bool
    run_pexeso: bool
    run_optional_finetune: bool
    quick_mode: bool
    short_query_lengths_seconds: tuple[float, ...]
    multi_segment_group_size: int
    search_oversample_factor: int
    faiss_nprobe_values: tuple[int, ...]
    active_training_policy_names: tuple[str, ...]
    active_control_run_ids: tuple[str, ...]
    active_query_regime_names: tuple[QueryRegimeName, ...]
    active_windowing_names: tuple[WindowingSpecName, ...]
    active_index_names: tuple[str, ...]
    pexeso_root: Path | None = None
    optional_partial_unfreeze_last_layers: int = 2

    @property
    def segment_samples(self) -> int:
        return int(self.segment_seconds * self.sample_rate)

    @property
    def spectrogram_frames(self) -> int:
        return 1 + self.segment_samples // self.hop_length

    @property
    def device(self) -> torch.device:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    @property
    def device_type(self) -> str:
        return "cuda" if torch.cuda.is_available() else "cpu"


def build_runtime_info() -> RuntimeInfo:
    """Capture runtime properties that influence notebook policy."""
    return RuntimeInfo(
        python_version=platform.python_version(),
        torch_version=torch.__version__,
        torchaudio_version=torchaudio.__version__,
        transformers_version=__import__("transformers").__version__,
        faiss_version=faiss.__version__,
        cuda_available=bool(torch.cuda.is_available()),
        device_name=get_gpu_name(),
        total_vram_gb=get_total_vram_gb(),
        bf16_supported=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    )


def build_default_config(runtime_info: RuntimeInfo) -> ExperimentConfig:
    """Construct the default configuration."""
    output_root = Path("/content/song_fingerprinting_outputs/notebook3_robustness")
    paths = PathConfig(
        data_root=Path("/content/data"),
        output_root=output_root,
        drive_mount_point=Path("/content/drive"),
        cache_dir=output_root / "cache",
        checkpoint_dir=output_root / "checkpoints",
        embedding_cache_dir=output_root / "embeddings",
        index_dir=output_root / "indexes",
        metrics_dir=output_root / "metrics",
        plot_dir=output_root / "plots",
        export_dir=output_root / "exports",
        log_dir=output_root / "logs",
    )
    run_presets = (
        ModelRunConfig("hybrid_transformer", "filters_only", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "one_of_k", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "severity_controlled", 128, 6, 1e-4, 1e-4, 0.07, False, True),
        ModelRunConfig("hybrid_transformer", "two_of_k", 128, 6, 1e-4, 1e-4, 0.07, False, False),
        ModelRunConfig("hybrid_transformer", "curriculum_filters_late", 128, 6, 1e-4, 1e-4, 0.07, False, False),
    )
    return ExperimentConfig(
        metadata_url="https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        audio_url="https://os.unil.cloud.switch.ch/fma/fma_small.zip",
        paths=paths,
        seed=42,
        batch_size=24,
        mert_batch_size=8,
        num_workers=2,
        sample_rate=16_000,
        n_fft=1024,
        hop_length=256,
        n_mels=128,
        segment_seconds=3.0,
        amp_enabled=runtime_info.cuda_available,
        gradient_checkpointing=False,
        early_stopping_patience=3,
        top_k=(1, 5, 10),
        projection_hidden_dim=1024,
        max_train_tracks=None,
        max_eval_tracks=None,
        smoke_test_track_count=16,
        gradient_clip_norm=1.0,
        run_presets=run_presets,
        strict_artifact_validation=True,
        mount_google_drive=True,
        save_to_drive=True,
        allow_retrain_missing_baselines=False,
        use_torch_compile=False,
        run_training_ablations=True,
        run_faiss_sweep=True,
        run_error_analysis=True,
        run_pexeso=False,
        run_optional_finetune=False,
        quick_mode=False,
        short_query_lengths_seconds=(1.0, 2.0, 3.0),
        multi_segment_group_size=3,
        search_oversample_factor=16,
        faiss_nprobe_values=(1, 4, 8, 16),
        active_training_policy_names=("filters_only", "one_of_k", "severity_controlled"),
        active_control_run_ids=(
            "cnn_baseline_embed128",
            "hybrid_transformer_baseline_embed128",
            "hybrid_transformer_extended_embed128",
            "hybrid_transformer_extended_embed256",
            "frozen_mert_extended_embed128",
        ),
        active_query_regime_names=(
            "clean_current",
            "short_centered",
            "short_offcenter",
            "combined_moderate",
            "multi_segment_same_track",
        ),
        active_windowing_names=("single_center", "multi3_even"),
        active_index_names=("exact_ip", "ivfflat_nprobe1", "ivfflat_nprobe4", "ivfflat_nprobe8", "ivfpq_nprobe4"),
    )


RUNTIME_INFO = build_runtime_info()
NOTEBOOK3_CONFIG = build_default_config(RUNTIME_INFO)
CONFIG = NOTEBOOK3_CONFIG

for output_directory in CONFIG.paths.all_directories():
    output_directory.mkdir(parents=True, exist_ok=True)

CONFIG_OVERVIEW = pd.DataFrame(
    [
        {"field": "device", "value": str(CONFIG.device)},
        {"field": "gpu_name", "value": RUNTIME_INFO.device_name},
        {"field": "vram_gb", "value": f"{RUNTIME_INFO.total_vram_gb:.2f}"},
        {"field": "amp_enabled", "value": str(CONFIG.amp_enabled)},
        {"field": "amp_dtype", "value": "bfloat16" if RUNTIME_INFO.bf16_supported else "float16"},
        {"field": "output_root", "value": str(CONFIG.paths.output_root)},
        {"field": "run_training_ablations", "value": str(CONFIG.run_training_ablations)},
        {"field": "run_faiss_sweep", "value": str(CONFIG.run_faiss_sweep)},
        {"field": "run_pexeso", "value": str(CONFIG.run_pexeso)},
    ]
)
display(CONFIG_OVERVIEW)


## Environment, Utilities, And Artifact Discovery

This section:
- provides the shared utility layer used by the rest of the notebook
- optionally mounts Google Drive when explicitly enabled
- validates the Notebook 2 artifacts that this notebook depends on

Missing core transformer checkpoints should fail loudly instead of being silently ignored.


In [ ]:
# Shared utilities and Notebook 2 artifact discovery.
def serialize_jsonable(value: object) -> object:
    """Recursively convert notebook objects into JSON-safe values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, list):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): serialize_jsonable(item) for key, item in value.items()}
    if hasattr(value, "__dataclass_fields__"):
        return {
            field_name: serialize_jsonable(getattr(value, field_name))
            for field_name in value.__dataclass_fields__.keys()
        }
    return value


def safe_mkdir(path: Path) -> Path:
    """Create a directory and return the resolved path."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_json(filepath: Path, payload: object) -> None:
    """Write a JSON artifact with stable indentation."""
    safe_mkdir(filepath.parent)
    with filepath.open("w", encoding="utf-8") as handle:
        json.dump(serialize_jsonable(payload), handle, indent=2)
        handle.write("\n")


def load_json(filepath: Path) -> object:
    """Load a JSON artifact from disk."""
    with filepath.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_pickle(filepath: Path, payload: object) -> None:
    """Persist an object using pickle for cache-heavy notebook phases."""
    safe_mkdir(filepath.parent)
    with filepath.open("wb") as handle:
        pickle.dump(payload, handle)


def load_pickle(filepath: Path) -> object:
    """Load a pickle payload from disk."""
    with filepath.open("rb") as handle:
        return pickle.load(handle)


def save_dataframe(filepath: Path, dataframe: pd.DataFrame) -> None:
    """Write a dataframe to CSV after ensuring the parent directory exists."""
    safe_mkdir(filepath.parent)
    dataframe.to_csv(filepath, index=False)


def checkpoint_exists(path: Path | None) -> bool:
    """Return whether a checkpoint path exists and is a regular file."""
    return bool(path is not None and path.is_file())


def normalize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """L2-normalize a float32 embedding matrix without mutating the caller's array."""
    matrix = np.asarray(embeddings, dtype=np.float32).copy()
    faiss.normalize_L2(matrix)
    return matrix


def safe_to_device(batch: object, device: torch.device, non_blocking: bool = True) -> object:
    """Move nested tensors to a device while leaving non-tensors untouched."""
    if isinstance(batch, torch.Tensor):
        return batch.to(device, non_blocking=non_blocking)
    if isinstance(batch, dict):
        return {
            key: safe_to_device(value, device, non_blocking=non_blocking)
            for key, value in batch.items()
        }
    if isinstance(batch, list):
        return [safe_to_device(value, device, non_blocking=non_blocking) for value in batch]
    return batch


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    """Count model parameters, optionally restricting to trainable tensors."""
    parameters = model.parameters()
    if trainable_only:
        parameters = (parameter for parameter in parameters if parameter.requires_grad)
    return int(sum(parameter.numel() for parameter in parameters))


def format_seconds(total_seconds: float) -> str:
    """Format a duration in seconds into a compact human-readable string."""
    total_seconds = max(0.0, float(total_seconds))
    minutes, seconds = divmod(int(total_seconds), 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def hash_experiment_config(payload: object) -> str:
    """Hash a JSON-serializable payload into a short stable cache key."""
    encoded = json.dumps(serialize_jsonable(payload), sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def dataframe_hash(dataframe: pd.DataFrame) -> str:
    """Create a stable hash for a dataframe used in notebook caches."""
    hashed_values = pd.util.hash_pandas_object(dataframe, index=True).values.tobytes()
    return hashlib.sha256(hashed_values).hexdigest()[:16]


def print_phase_banner(title: str) -> None:
    """Print a compact phase banner for long-running notebook stages."""
    print(f"\n{'=' * 20} {title} {'=' * 20}")


def clear_memory() -> None:
    """Release cached Python and CUDA memory between notebook phases."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def maybe_mount_google_drive(config: ExperimentConfig) -> Path | None:
    """Mount Google Drive when explicitly enabled and available in the runtime."""
    if not config.mount_google_drive:
        print("Google Drive mounting is disabled by configuration.")
        return None
    try:
        from google.colab import drive as colab_drive
    except ImportError:
        print("google.colab is unavailable; continuing without Drive.")
        return None
    mount_point = config.paths.drive_mount_point
    colab_drive.mount(str(mount_point))
    return mount_point


def resolve_notebook2_output_roots(config: ExperimentConfig, drive_root: Path | None) -> list[Path]:
    """Resolve plausible Notebook 2 output roots from local and Drive-backed locations."""
    candidate_roots = [Path("/content/song_fingerprinting_outputs")]

    if drive_root is not None:
        candidate_roots.extend(
            [
                drive_root / "MyDrive" / "song_fingerprinting_outputs",
                drive_root / "MyDrive" / "transformer-fingerprinting" / "song_fingerprinting_outputs",
                drive_root / "MyDrive" / "Colab Notebooks" / "song_fingerprinting_outputs",
            ]
        )

        notebook2_export_root = drive_root / "MyDrive" / "transformer_fingerprinting" / "artifacts" / "notebook2"
        if notebook2_export_root.exists():
            for run_export_dir in sorted(notebook2_export_root.glob("run_*")):
                exported_output_root = run_export_dir / "song_fingerprinting_outputs"
                if exported_output_root.exists():
                    candidate_roots.append(exported_output_root)

    unique_roots: list[Path] = []
    seen_roots: set[str] = set()
    for root in candidate_roots:
        root_str = str(root)
        if root_str in seen_roots:
            continue
        seen_roots.add(root_str)
        unique_roots.append(root)

    return unique_roots


NOTEBOOK2_RUN_REQUIREMENTS: dict[str, dict[str, object]] = {
    "hybrid_transformer_baseline_embed128": {
        "required": True,
        "model_name": "hybrid_transformer",
        "augmentation_profile": "baseline",
        "embed_dim": 128,
        "freeze_backbone": False,
    },
    "hybrid_transformer_extended_embed128": {
        "required": True,
        "model_name": "hybrid_transformer",
        "augmentation_profile": "extended_existing",
        "embed_dim": 128,
        "freeze_backbone": False,
    },
    "cnn_baseline_embed128": {
        "required": False,
        "model_name": "cnn",
        "augmentation_profile": "baseline",
        "embed_dim": 128,
        "freeze_backbone": False,
    },
    "hybrid_transformer_extended_embed256": {
        "required": False,
        "model_name": "hybrid_transformer",
        "augmentation_profile": "extended_existing",
        "embed_dim": 256,
        "freeze_backbone": False,
    },
    "frozen_mert_extended_embed128": {
        "required": False,
        "model_name": "frozen_mert",
        "augmentation_profile": "extended_existing",
        "embed_dim": 128,
        "freeze_backbone": True,
    },
}


def discover_notebook2_artifacts(config: ExperimentConfig, candidate_roots: list[Path]) -> pd.DataFrame:
    """Validate the Notebook 2 artifacts that this notebook depends on."""
    rows: list[dict[str, object]] = []
    for run_id, metadata in NOTEBOOK2_RUN_REQUIREMENTS.items():
        discovered_root: Path | None = None
        for root in candidate_roots:
            run_dir = root / run_id
            if run_dir.exists():
                discovered_root = run_dir
                break

        checkpoint_path = discovered_root / "checkpoint_best.pt" if discovered_root is not None else None
        metrics_path = discovered_root / "metrics.csv" if discovered_root is not None else None
        config_path = discovered_root / "config.json" if discovered_root is not None else None
        degradation_path = discovered_root / "degradation_breakdown.csv" if discovered_root is not None else None
        rows.append(
            {
                "run_id": run_id,
                "required": bool(metadata["required"]),
                "source_root": str(discovered_root) if discovered_root is not None else "",
                "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else "",
                "checkpoint_exists": checkpoint_exists(checkpoint_path),
                "metrics_path": str(metrics_path) if metrics_path is not None else "",
                "metrics_exists": bool(metrics_path is not None and metrics_path.exists()),
                "config_path": str(config_path) if config_path is not None else "",
                "config_exists": bool(config_path is not None and config_path.exists()),
                "degradation_path": str(degradation_path) if degradation_path is not None else "",
                "degradation_exists": bool(degradation_path is not None and degradation_path.exists()),
                "model_name": str(metadata["model_name"]),
                "augmentation_profile": str(metadata["augmentation_profile"]),
                "embed_dim": int(metadata["embed_dim"]),
                "freeze_backbone": bool(metadata["freeze_backbone"]),
            }
        )
    artifact_df = pd.DataFrame(rows).sort_values(["required", "run_id"], ascending=[False, True]).reset_index(drop=True)
    missing_required = artifact_df[(artifact_df["required"]) & (~artifact_df["checkpoint_exists"])]
    if not missing_required.empty and config.strict_artifact_validation and not config.allow_retrain_missing_baselines:
        missing_run_ids = ", ".join(missing_required["run_id"].tolist())
        raise FileNotFoundError(
            "Required Notebook 2 checkpoints were not found. "
            f"Missing runs: {missing_run_ids}. "
            "Enable a Drive mirror or set allow_retrain_missing_baselines=True before continuing."
        )
    return artifact_df


print_phase_banner("Runtime summary and Notebook 2 artifact discovery")
DRIVE_ROOT = maybe_mount_google_drive(CONFIG)
NOTEBOOK2_OUTPUT_ROOTS = resolve_notebook2_output_roots(CONFIG, DRIVE_ROOT)
NOTEBOOK2_ARTIFACTS_DF = discover_notebook2_artifacts(CONFIG, NOTEBOOK2_OUTPUT_ROOTS)
display(NOTEBOOK2_ARTIFACTS_DF)
save_json(
    CONFIG.paths.export_dir / "resolved_notebook3_config.json",
    {"runtime": RUNTIME_INFO, "config": CONFIG, "notebook2_output_roots": NOTEBOOK2_OUTPUT_ROOTS},
)


## Dataset Bootstrap

The notebook owns data bootstrap because Colab runtimes are ephemeral. This section downloads `fma_metadata.zip` and `fma_small.zip`, extracts them into `/content/data`, loads `tracks.csv`, and verifies the track-level split plus audio-file availability before any training code is allowed to run.

In [ ]:
# Dataset bootstrap helpers.
def run_command(command: list[str], cwd: Path | None = None) -> None:
    """Run a subprocess command and surface failures immediately."""
    subprocess.run(command, cwd=str(cwd) if cwd is not None else None, check=True)


def download_file(url: str, destination: Path) -> None:
    """Download a file with curl and overwrite any partial artifact."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {destination}")
    run_command(["curl", "-L", "--fail", "--output", str(destination), url])


def extract_zip_file(zip_path: Path, destination_dir: Path) -> None:
    """Extract a zip archive into the requested directory."""
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip archive not found: {zip_path}")
    destination_dir.mkdir(parents=True, exist_ok=True)
    run_command(["unzip", "-q", "-o", str(zip_path), "-d", str(destination_dir)])


def download_fma_assets(config: ExperimentConfig) -> None:
    """Ensure FMA metadata and FMA Small audio are available locally."""
    if not config.paths.metadata_dir.exists():
        download_file(config.metadata_url, config.paths.metadata_zip)
        extract_zip_file(config.paths.metadata_zip, config.paths.data_root)
    else:
        print(f"Metadata already present at {config.paths.metadata_dir}")

    if not config.paths.audio_dir.exists():
        download_file(config.audio_url, config.paths.audio_zip)
        extract_zip_file(config.paths.audio_zip, config.paths.data_root)
    else:
        print(f"Audio already present at {config.paths.audio_dir}")


def load_tracks(filepath: Path) -> pd.DataFrame:
    """Load `tracks.csv` using the official MultiIndex column structure."""
    tracks = pd.read_csv(filepath, index_col=0, header=[0, 1])

    list_columns = [
        ("track", "tags"),
        ("track", "genres"),
        ("track", "genres_all"),
    ]
    for column in list_columns:
        tracks[column] = tracks[column].map(
            lambda value: ast.literal_eval(value) if isinstance(value, str) else value
        )

    datetime_columns = [
        ("track", "date_created"),
        ("track", "date_recorded"),
        ("album", "date_created"),
        ("album", "date_released"),
        ("artist", "date_created"),
        ("artist", "active_year_begin"),
        ("artist", "active_year_end"),
    ]
    for column in datetime_columns:
        tracks[column] = pd.to_datetime(tracks[column], errors="coerce")

    subset_dtype = pd.CategoricalDtype(categories=("small", "medium", "large"), ordered=True)
    split_dtype = pd.CategoricalDtype(categories=("training", "validation", "test"), ordered=True)
    tracks["set", "subset"] = tracks["set", "subset"].astype(subset_dtype)
    tracks["set", "split"] = tracks["set", "split"].astype(split_dtype)
    return tracks


def load_metadata_tables(config: ExperimentConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the FMA metadata tables required by the experiment notebook."""
    metadata_dir = config.paths.metadata_dir
    tracks = load_tracks(metadata_dir / "tracks.csv")
    genres = pd.read_csv(metadata_dir / "genres.csv", index_col=0)
    features = pd.read_csv(metadata_dir / "features.csv", index_col=0, header=[0, 1, 2])
    return tracks, genres, features


def get_audio_path(audio_dir: Path, track_id: int) -> Path:
    """Build the FMA Small file path for a track ID."""
    track_key = f"{track_id:06d}"
    return audio_dir / track_key[:3] / f"{track_key}.mp3"


def verify_dataset_layout(audio_dir: Path, subset_tracks: pd.DataFrame) -> dict[str, int]:
    """Check the expected FMA Small layout and count missing audio files."""
    if not audio_dir.exists():
        raise FileNotFoundError(f"Audio directory does not exist: {audio_dir}")

    subset_ids = [int(track_id) for track_id in subset_tracks.index.tolist()]
    missing_count = sum(0 if get_audio_path(audio_dir, track_id).exists() else 1 for track_id in subset_ids)
    return {
        "total_tracks": len(subset_ids),
        "training_tracks": int((subset_tracks["set", "split"] == "training").sum()),
        "validation_tracks": int((subset_tracks["set", "split"] == "validation").sum()),
        "test_tracks": int((subset_tracks["set", "split"] == "test").sum()),
        "missing_audio_files": missing_count,
    }

def find_undecodable_audio_tracks(
    audio_dir: Path,
    track_ids: list[int],
    probe_sample_rate: int,
) -> pd.DataFrame:
    """Return tracks that exist on disk but fail to decode in the current runtime."""
    import warnings

    failure_rows: list[dict[str, object]] = []

    for track_id in tqdm(track_ids, desc="probe-audio", leave=False):
        filepath = get_audio_path(audio_dir, track_id)

        if not filepath.is_file():
            failure_rows.append(
                {
                    "track_id": int(track_id),
                    "path": str(filepath),
                    "error_type": "FileNotFoundError",
                    "error_message": "File does not exist or is not a regular file.",
                }
            )
            continue

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                # Short probe is enough to catch the known corrupted files early.
                librosa.load(str(filepath), sr=probe_sample_rate, mono=True, duration=0.1)
        except Exception as exc:
            failure_rows.append(
                {
                    "track_id": int(track_id),
                    "path": str(filepath),
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }
            )

    return pd.DataFrame(failure_rows)


In [ ]:
download_fma_assets(CONFIG)
tracks, genres, features = load_metadata_tables(CONFIG)
tracks_small = tracks[tracks["set", "subset"] <= "small"].copy()

audio_probe_failures = find_undecodable_audio_tracks(
    audio_dir=CONFIG.paths.audio_dir,
    track_ids=[int(track_id) for track_id in tracks_small.index.tolist()],
    probe_sample_rate=CONFIG.sample_rate,
)

known_bad_ids = {99134, 108925, 133297}
detected_bad_ids = set(audio_probe_failures["track_id"].tolist()) if not audio_probe_failures.empty else set()
excluded_track_ids = sorted(known_bad_ids | detected_bad_ids)

tracks_small = tracks_small.loc[~tracks_small.index.isin(excluded_track_ids)].copy()

dataset_summary = verify_dataset_layout(CONFIG.paths.audio_dir, tracks_small)
dataset_summary["excluded_bad_audio_files"] = len(excluded_track_ids)

split_distribution = tracks_small["set", "split"].value_counts().rename_axis("split").reset_index(name="tracks")
genre_coverage = (
    tracks_small.groupby([("set", "split"), ("track", "genre_top")])
    .size()
    .rename("tracks")
    .reset_index()
    .rename(columns={("set", "split"): "split", ("track", "genre_top"): "genre"})
)

print("Dataset summary:")
print(json.dumps(dataset_summary, indent=2))
print(f"Excluded bad track IDs: {excluded_track_ids}")

if not audio_probe_failures.empty:
    display(audio_probe_failures.sort_values("track_id").reset_index(drop=True))

display(split_distribution)
display(genre_coverage.pivot(index="genre", columns="split", values="tracks").fillna(0).astype(int))


## Preprocessing

Reuse the working 16 kHz log-mel path from the exploration notebook for the CNN and hybrid transformer. MERT uses a separate raw-waveform path and its own processor sample rate. Every helper here validates inputs and returns deterministic shapes so later training cells do not depend on hidden notebook state.

### Audio loading, segmentation, normalization, and spectrogram helpers

In [ ]:
def validate_positive_int(name: str, value: int) -> None:
    """Validate that an integer configuration value is strictly positive."""
    if value <= 0:
        raise ValueError(f"{name} must be positive, received {value}.")


def validate_non_empty_waveform(waveform: np.ndarray) -> None:
    """Validate that an audio array contains at least one sample."""
    if waveform.ndim != 1:
        raise ValueError(f"Waveform must be 1-D, received shape {waveform.shape}.")
    if waveform.size == 0:
        raise ValueError("Waveform is empty.")


def load_audio_file(filepath: Path, target_sr: int | None, mono: bool = True) -> tuple[np.ndarray, int]:
    """Load an audio file from disk and optionally resample it."""
    if not filepath.exists():
        raise FileNotFoundError(f"Audio file not found: {filepath}")
    waveform, sample_rate = librosa.load(str(filepath), sr=target_sr, mono=mono)
    validate_non_empty_waveform(waveform)
    return waveform.astype(np.float32), int(target_sr if target_sr is not None else sample_rate)


def trim_or_pad_waveform(waveform: np.ndarray, target_length: int) -> np.ndarray:
    """Return a waveform with exactly `target_length` samples."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] >= target_length:
        return waveform[:target_length].astype(np.float32)
    padded = np.zeros(target_length, dtype=np.float32)
    padded[: waveform.shape[0]] = waveform
    return padded


def normalize_waveform(waveform: np.ndarray) -> np.ndarray:
    """Peak-normalize a waveform while guarding against silent inputs."""
    validate_non_empty_waveform(waveform)
    peak = float(np.max(np.abs(waveform)))
    if peak < 1e-8:
        return waveform.astype(np.float32)
    return (waveform / peak).astype(np.float32)


def extract_random_segment(waveform: np.ndarray, target_length: int, rng: np.random.Generator) -> tuple[np.ndarray, int]:
    """Sample a random contiguous segment and return the segment plus its start index."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] <= target_length:
        return trim_or_pad_waveform(waveform, target_length), 0
    max_start = waveform.shape[0] - target_length
    start_index = int(rng.integers(0, max_start + 1))
    segment = waveform[start_index : start_index + target_length]
    return segment.astype(np.float32), start_index


def extract_center_segment(waveform: np.ndarray, target_length: int) -> tuple[np.ndarray, int]:
    """Extract the centered segment used for deterministic retrieval evaluation."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("target_length", target_length)
    if waveform.shape[0] <= target_length:
        return trim_or_pad_waveform(waveform, target_length), 0
    start_index = (waveform.shape[0] - target_length) // 2
    segment = waveform[start_index : start_index + target_length]
    return segment.astype(np.float32), start_index


def compute_log_mel_spectrogram(
    waveform: np.ndarray,
    sample_rate: int,
    n_fft: int,
    hop_length: int,
    n_mels: int,
) -> np.ndarray:
    """Convert a raw waveform into a log-mel spectrogram."""
    validate_non_empty_waveform(waveform)
    validate_positive_int("sample_rate", sample_rate)
    validate_positive_int("n_fft", n_fft)
    validate_positive_int("hop_length", hop_length)
    validate_positive_int("n_mels", n_mels)
    mel = librosa.feature.melspectrogram(
        y=waveform,
        sr=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def normalize_spectrogram(spectrogram: np.ndarray) -> np.ndarray:
    """Standardize a spectrogram on a per-sample basis."""
    if spectrogram.ndim != 2:
        raise ValueError(f"Spectrogram must be 2-D, received shape {spectrogram.shape}.")
    mean = float(spectrogram.mean())
    std = float(spectrogram.std())
    if std < 1e-6:
        return (spectrogram - mean).astype(np.float32)
    return ((spectrogram - mean) / std).astype(np.float32)


def spectrogram_tensor_from_waveform(waveform: np.ndarray, config: ExperimentConfig) -> torch.Tensor:
    """Create a normalized `(1, n_mels, time_frames)` tensor from a waveform segment."""
    spectrogram = compute_log_mel_spectrogram(
        waveform=waveform,
        sample_rate=config.sample_rate,
        n_fft=config.n_fft,
        hop_length=config.hop_length,
        n_mels=config.n_mels,
    )
    spectrogram = normalize_spectrogram(spectrogram)
    return torch.from_numpy(spectrogram).unsqueeze(0).float()


## Augmentations

Two augmentation presets are kept on purpose:
- `baseline`: random gain, additive noise, mild time stretch, and mild pitch shift
- `extended`: the baseline stack plus low-pass and high-pass filtering

Every augmentation validates its parameters and trims or pads back to the configured segment length immediately after mutation.

### Augmentation helpers and visualization

In [ ]:
def add_gaussian_noise(waveform: np.ndarray, rng: np.random.Generator, snr_db: float) -> np.ndarray:
    """Add Gaussian noise at a specified signal-to-noise ratio."""
    if snr_db <= 0:
        raise ValueError(f"snr_db must be positive, received {snr_db}.")
    signal_power = float(np.mean(np.square(waveform)))
    noise_power = signal_power / (10.0 ** (snr_db / 10.0))
    noise = rng.normal(0.0, np.sqrt(noise_power), size=waveform.shape[0]).astype(np.float32)
    return (waveform + noise).astype(np.float32)


def apply_random_gain(waveform: np.ndarray, rng: np.random.Generator, min_db: float, max_db: float) -> np.ndarray:
    """Apply a random volume change in decibels."""
    if min_db > max_db:
        raise ValueError("min_db must be <= max_db.")
    gain_db = float(rng.uniform(min_db, max_db))
    gain = 10.0 ** (gain_db / 20.0)
    return (waveform * gain).astype(np.float32)


def apply_time_stretch(waveform: np.ndarray, rate: float) -> np.ndarray:
    """Time-stretch a waveform without changing pitch."""
    if rate <= 0:
        raise ValueError(f"rate must be positive, received {rate}.")
    return librosa.effects.time_stretch(waveform, rate=rate).astype(np.float32)


def apply_pitch_shift(waveform: np.ndarray, sample_rate: int, n_steps: float) -> np.ndarray:
    """Pitch-shift a waveform by the requested number of semitones."""
    return librosa.effects.pitch_shift(waveform, sr=sample_rate, n_steps=n_steps).astype(np.float32)


def apply_lowpass_filter(waveform: np.ndarray, sample_rate: int, cutoff_hz: float, order: int = 5) -> np.ndarray:
    """Apply a Butterworth low-pass filter."""
    nyquist = sample_rate / 2.0
    if cutoff_hz <= 0 or cutoff_hz >= nyquist:
        raise ValueError(f"cutoff_hz must be in (0, {nyquist}), received {cutoff_hz}.")
    b, a = scipy.signal.butter(order, cutoff_hz / nyquist, btype="low")
    return scipy.signal.filtfilt(b, a, waveform).astype(np.float32)


def apply_highpass_filter(waveform: np.ndarray, sample_rate: int, cutoff_hz: float, order: int = 5) -> np.ndarray:
    """Apply a Butterworth high-pass filter."""
    nyquist = sample_rate / 2.0
    if cutoff_hz <= 0 or cutoff_hz >= nyquist:
        raise ValueError(f"cutoff_hz must be in (0, {nyquist}), received {cutoff_hz}.")
    b, a = scipy.signal.butter(order, cutoff_hz / nyquist, btype="high")
    return scipy.signal.filtfilt(b, a, waveform).astype(np.float32)


def apply_augmentation_profile(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    profile: AugmentationProfileName,
    rng: np.random.Generator,
) -> np.ndarray:
    """Apply the selected augmentation profile to a waveform segment."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    segment = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-6.0, max_db=6.0), target_length)
    segment = trim_or_pad_waveform(add_gaussian_noise(segment, rng=rng, snr_db=float(rng.uniform(10.0, 25.0))), target_length)
    segment = trim_or_pad_waveform(apply_time_stretch(segment, rate=float(rng.uniform(0.92, 1.08))), target_length)
    segment = trim_or_pad_waveform(apply_pitch_shift(segment, sample_rate=sample_rate, n_steps=float(rng.uniform(-2.0, 2.0))), target_length)

    if profile == "extended":
        segment = trim_or_pad_waveform(
            apply_lowpass_filter(segment, sample_rate=sample_rate, cutoff_hz=float(rng.uniform(2500.0, 5000.0))),
            target_length,
        )
        segment = trim_or_pad_waveform(
            apply_highpass_filter(segment, sample_rate=sample_rate, cutoff_hz=float(rng.uniform(120.0, 900.0))),
            target_length,
        )

    return normalize_waveform(segment)


def apply_retrieval_degradation(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    condition: RetrievalConditionName,
    rng: np.random.Generator,
) -> np.ndarray:
    """Apply a deterministic query degradation for retrieval evaluation."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    if condition == "clean":
        return segment
    if condition == "noise":
        return trim_or_pad_waveform(add_gaussian_noise(segment, rng=rng, snr_db=8.0), target_length)
    if condition == "pitch":
        return trim_or_pad_waveform(apply_pitch_shift(segment, sample_rate=sample_rate, n_steps=2.0), target_length)
    if condition == "time":
        return trim_or_pad_waveform(apply_time_stretch(segment, rate=1.10), target_length)
    if condition == "lowpass":
        return trim_or_pad_waveform(apply_lowpass_filter(segment, sample_rate=sample_rate, cutoff_hz=3000.0), target_length)
    if condition == "highpass":
        return trim_or_pad_waveform(apply_highpass_filter(segment, sample_rate=sample_rate, cutoff_hz=500.0), target_length)
    raise ValueError(f"Unsupported retrieval condition: {condition}")


def visualize_preprocessing_example(config: ExperimentConfig, tracks_subset: pd.DataFrame) -> None:
    """Plot one clean segment and one extended-augmentation segment for sanity checking."""
    example_track_id = int(tracks_subset.index[0])
    waveform, _ = load_audio_file(get_audio_path(config.paths.audio_dir, example_track_id), target_sr=config.sample_rate)
    clean_segment, _ = extract_center_segment(waveform, config.segment_samples)
    augmented_segment = apply_augmentation_profile(
        waveform=clean_segment,
        sample_rate=config.sample_rate,
        target_length=config.segment_samples,
        profile="extended",
        rng=np.random.default_rng(config.seed),
    )
    clean_spec = compute_log_mel_spectrogram(clean_segment, config.sample_rate, config.n_fft, config.hop_length, config.n_mels)
    augmented_spec = compute_log_mel_spectrogram(augmented_segment, config.sample_rate, config.n_fft, config.hop_length, config.n_mels)

    fig, axes = plt.subplots(2, 2, figsize=(16, 8))
    axes[0, 0].plot(clean_segment)
    axes[0, 0].set_title(f"Clean waveform | track {example_track_id}")
    axes[0, 1].plot(augmented_segment)
    axes[0, 1].set_title("Extended-augmentation waveform")
    axes[1, 0].imshow(clean_spec, origin="lower", aspect="auto", cmap="magma")
    axes[1, 0].set_title("Clean log-mel spectrogram")
    axes[1, 1].imshow(augmented_spec, origin="lower", aspect="auto", cmap="magma")
    axes[1, 1].set_title("Augmented log-mel spectrogram")
    plt.tight_layout()
    plt.show()


visualize_preprocessing_example(CONFIG, tracks_small)


## Datasets And Loaders

The training set yields anchor and positive pairs for contrastive learning. Retrieval datasets produce one deterministic reference or query segment per track. Spectrogram models receive `(1, n_mels, time_frames)` tensors, while MERT receives raw waveforms and uses a custom collate function built around `Wav2Vec2FeatureExtractor`.

### Dataset classes, collate functions, and split helpers

In [ ]:
class PairBatch(TypedDict):
    anchor: torch.Tensor
    positive: torch.Tensor
    track_id: torch.Tensor
    anchor_start_time: torch.Tensor


class RetrievalBatch(TypedDict):
    inputs: torch.Tensor
    track_id: torch.Tensor


def seed_worker(worker_id: int) -> None:
    """Seed NumPy and Python RNGs inside a DataLoader worker."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_torch_generator(seed: int) -> torch.Generator:
    """Build a deterministic PyTorch generator for DataLoader shuffling."""
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


class ContrastivePairDataset(Dataset[dict[str, object]]):
    """Yield anchor and positive pairs for contrastive training."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        augmentation_profile: AugmentationProfileName,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.augmentation_profile = augmentation_profile

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        seed = int(torch.randint(low=0, high=2**31 - 1, size=(1,)).item())
        rng = np.random.default_rng(seed)
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, track_id), target_sr=self.sample_rate)
        segment, start_index = extract_random_segment(waveform, self.segment_length, rng=rng)
        segment = normalize_waveform(trim_or_pad_waveform(segment, self.segment_length))
        positive_waveform = apply_augmentation_profile(
            waveform=segment.copy(),
            sample_rate=self.sample_rate,
            target_length=self.segment_length,
            profile=self.augmentation_profile,
            rng=rng,
        )

        if self.input_mode == "spectrogram":
            anchor_value: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
            positive_value: torch.Tensor = spectrogram_tensor_from_waveform(positive_waveform, self.config)
        else:
            anchor_value = torch.from_numpy(segment).float()
            positive_value = torch.from_numpy(positive_waveform).float()

        return {
            "anchor": anchor_value,
            "positive": positive_value,
            "track_id": int(track_id),
            "anchor_start_time": float(start_index / self.sample_rate),
        }


class RetrievalReferenceDataset(Dataset[dict[str, object]]):
    """Yield deterministic reference segments for FAISS indexing."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, track_id), target_sr=self.sample_rate)
        segment, _ = extract_center_segment(waveform, self.segment_length)
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
        else:
            model_input = torch.from_numpy(segment).float()
        return {"inputs": model_input, "track_id": int(track_id)}


class RetrievalQueryDataset(Dataset[dict[str, object]]):
    """Yield deterministic degraded query segments for retrieval evaluation."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        condition: RetrievalConditionName,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.condition = condition

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = self.track_ids[index]
        rng = np.random.default_rng(self.config.seed + index)
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, track_id), target_sr=self.sample_rate)
        segment, _ = extract_center_segment(waveform, self.segment_length)
        degraded = apply_retrieval_degradation(
            waveform=segment,
            sample_rate=self.sample_rate,
            target_length=self.segment_length,
            condition=self.condition,
            rng=rng,
        )
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(degraded, self.config)
        else:
            model_input = torch.from_numpy(degraded).float()
        return {"inputs": model_input, "track_id": int(track_id)}


def collate_pair_batch(batch: list[dict[str, object]]) -> PairBatch:
    """Stack spectrogram or waveform pair samples into one batch."""
    anchors = torch.stack([sample["anchor"] for sample in batch])
    positives = torch.stack([sample["positive"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    start_times = torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32)
    return {
        "anchor": anchors,
        "positive": positives,
        "track_id": track_ids,
        "anchor_start_time": start_times,
    }


def collate_retrieval_batch(batch: list[dict[str, object]]) -> RetrievalBatch:
    """Stack spectrogram or waveform retrieval samples into one batch."""
    inputs = torch.stack([sample["inputs"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    return {"inputs": inputs, "track_id": track_ids}


def build_mert_pair_collate_fn(feature_extractor: Wav2Vec2FeatureExtractor) -> Callable[[list[dict[str, object]]], dict[str, torch.Tensor]]:
    """Build the MERT pair collate function."""

    def collate(batch: list[dict[str, object]]) -> dict[str, torch.Tensor]:
        anchor_arrays = [np.asarray(sample["anchor"], dtype=np.float32) for sample in batch]
        positive_arrays = [np.asarray(sample["positive"], dtype=np.float32) for sample in batch]
        anchor_inputs = feature_extractor(anchor_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        positive_inputs = feature_extractor(positive_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        return {
            "anchor_input_values": anchor_inputs["input_values"],
            "anchor_attention_mask": anchor_inputs["attention_mask"],
            "positive_input_values": positive_inputs["input_values"],
            "positive_attention_mask": positive_inputs["attention_mask"],
            "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
            "anchor_start_time": torch.tensor([float(sample["anchor_start_time"]) for sample in batch], dtype=torch.float32),
        }

    return collate


def build_mert_retrieval_collate_fn(feature_extractor: Wav2Vec2FeatureExtractor) -> Callable[[list[dict[str, object]]], dict[str, torch.Tensor]]:
    """Build the MERT retrieval collate function."""

    def collate(batch: list[dict[str, object]]) -> dict[str, torch.Tensor]:
        waveform_arrays = [np.asarray(sample["inputs"], dtype=np.float32) for sample in batch]
        feature_batch = feature_extractor(waveform_arrays, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt", padding=True)
        return {
            "inputs": feature_batch["input_values"],
            "attention_mask": feature_batch["attention_mask"],
            "track_id": torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long),
        }

    return collate


def get_split_track_ids(
    subset_tracks: pd.DataFrame,
    max_train_tracks: int | None = None,
    max_eval_tracks: int | None = None,
) -> dict[str, list[int]]:
    """Collect track IDs for the training, validation, and test splits."""
    training_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "training"].index.tolist()]
    validation_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "validation"].index.tolist()]
    test_ids = [int(track_id) for track_id in subset_tracks[subset_tracks["set", "split"] == "test"].index.tolist()]

    if max_train_tracks is not None:
        training_ids = training_ids[:max_train_tracks]
    if max_eval_tracks is not None:
        validation_ids = validation_ids[:max_eval_tracks]
        test_ids = test_ids[:max_eval_tracks]

    return {
        "training": training_ids,
        "validation": validation_ids,
        "test": test_ids,
    }


SPLIT_TRACK_IDS = get_split_track_ids(
    tracks_small,
    max_train_tracks=CONFIG.max_train_tracks,
    max_eval_tracks=CONFIG.max_eval_tracks,
)
{split_name: len(track_ids) for split_name, track_ids in SPLIT_TRACK_IDS.items()}


## Minimal Baseline: Loss And Models

This section defines the shared NT-Xent loss, the compact CNN baseline, the hybrid spectrogram transformer, and the frozen MERT fingerprinter. All three models emit L2-normalized embeddings so FAISS can use exact inner-product search as cosine-equivalent retrieval.

### Loss and model definitions

In [ ]:
class NTXentLoss(nn.Module):
    """Symmetric NT-Xent loss for contrastive representation learning."""

    def __init__(self, temperature: float) -> None:
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        self.temperature = temperature

    def forward(self, anchor_embeddings: torch.Tensor, positive_embeddings: torch.Tensor) -> torch.Tensor:
        batch_size = anchor_embeddings.shape[0]
        if batch_size < 2:
            raise ValueError("NT-Xent requires a batch size of at least 2.")
        anchor_embeddings = F.normalize(anchor_embeddings, p=2, dim=1)
        positive_embeddings = F.normalize(positive_embeddings, p=2, dim=1)
        logits = anchor_embeddings @ positive_embeddings.T / self.temperature
        labels = torch.arange(batch_size, device=logits.device)
        loss_anchor = F.cross_entropy(logits, labels)
        loss_positive = F.cross_entropy(logits.T, labels)
        return 0.5 * (loss_anchor + loss_positive)


class ProjectionHead(nn.Module):
    """Two-layer projection head used by every fingerprinting model."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p=0.1),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        embeddings = self.layers(inputs)
        return F.normalize(embeddings, p=2, dim=1)


class ConvBlock(nn.Module):
    """A simple convolution, normalization, and activation block."""

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.block(inputs)


class CompactSpectrogramCNN(nn.Module):
    """Compact spectrogram CNN baseline for contrastive fingerprinting."""

    def __init__(self, embed_dim: int, projection_hidden_dim: int) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1, 32, stride=2),
            ConvBlock(32, 64, stride=2),
            ConvBlock(64, 128, stride=2),
            ConvBlock(128, 256, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.projection = ProjectionHead(256, projection_hidden_dim, embed_dim)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.encoder(inputs)
        pooled = self.pool(features).flatten(start_dim=1)
        return self.projection(pooled)


class TransformerEncoderStack(nn.Module):
    """A small Transformer encoder with optional activation checkpointing."""

    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float, use_gradient_checkpointing: bool) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=d_model,
                    nhead=num_heads,
                    dim_feedforward=d_model * 4,
                    dropout=dropout,
                    activation="gelu",
                    batch_first=True,
                    norm_first=True,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.use_gradient_checkpointing = use_gradient_checkpointing

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden_states = inputs
        for layer in self.layers:
            if self.use_gradient_checkpointing and self.training:
                hidden_states = checkpoint(lambda tensor: layer(tensor), hidden_states, use_reentrant=False)
            else:
                hidden_states = layer(hidden_states)
        return self.final_norm(hidden_states)


class HybridSpectrogramTransformer(nn.Module):
    """Hybrid spectrogram transformer used as the main scratch model."""

    def __init__(self, config: ExperimentConfig, embed_dim: int) -> None:
        super().__init__()
        self.config = config
        self.conv_stem = nn.Sequential(
            ConvBlock(1, 64, stride=2),
            ConvBlock(64, 128, stride=2),
        )
        self.patch_projection = nn.Conv2d(128, 256, kernel_size=4, stride=4)
        self.encoder = TransformerEncoderStack(256, 4, 4, 0.1, config.gradient_checkpointing)
        token_count = self._compute_token_count(config.n_mels, config.spectrogram_frames)
        self.position_embeddings = nn.Parameter(torch.zeros(1, token_count, 256))
        nn.init.trunc_normal_(self.position_embeddings, std=0.02)
        self.projection = ProjectionHead(256, config.projection_hidden_dim, embed_dim)

    @staticmethod
    def _compute_token_count(input_height: int, input_width: int) -> int:
        """Compute token count after the conv stem and patch projection."""
        height_after_stem = (input_height + 1) // 2
        height_after_stem = (height_after_stem + 1) // 2
        width_after_stem = (input_width + 1) // 2
        width_after_stem = (width_after_stem + 1) // 2
        height_after_patch = ((height_after_stem - 4) // 4) + 1
        width_after_patch = ((width_after_stem - 4) // 4) + 1
        return height_after_patch * width_after_patch

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.conv_stem(inputs)
        hidden = self.patch_projection(hidden)
        tokens = hidden.flatten(start_dim=2).transpose(1, 2)
        tokens = tokens + self.position_embeddings[:, : tokens.shape[1], :]
        encoded = self.encoder(tokens)
        pooled = encoded.mean(dim=1)
        return self.projection(pooled)


class FrozenMertFingerprinter(nn.Module):
    """Frozen MERT backbone with a trainable projection head."""

    def __init__(self, model_name: str, embed_dim: int, projection_hidden_dim: int, freeze_backbone: bool = True, weighted_layer_pooling: bool = False) -> None:
        super().__init__()
        self.model_name = model_name
        self.freeze_backbone = freeze_backbone
        self.weighted_layer_pooling = weighted_layer_pooling
        self.feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.hidden_size = int(self.backbone.config.hidden_size)
        self.projection = ProjectionHead(self.hidden_size, projection_hidden_dim, embed_dim)

        if self.weighted_layer_pooling:
            layer_count = int(self.backbone.config.num_hidden_layers) + 1
            self.layer_weights = nn.Parameter(torch.ones(layer_count) / layer_count)
        else:
            self.layer_weights = None

        if self.freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False
            self.backbone.eval()

    @property
    def sample_rate(self) -> int:
        return int(self.feature_extractor.sampling_rate)

    def train(self, mode: bool = True) -> "FrozenMertFingerprinter":
        super().train(mode)
        if self.freeze_backbone:
            self.backbone.eval()
        return self

    def _pool_hidden_states(self, outputs: object, attention_mask: torch.Tensor | None) -> torch.Tensor:
        """Pool hidden states with attention-mask awareness.

        For MERT/Hubert-style backbones, the raw waveform attention mask is longer
        than the encoder sequence length because the convolutional feature extractor
        downsamples the time axis. In that case, reduce the mask to feature-vector
        length before masked pooling.
        """
        if self.weighted_layer_pooling:
            hidden_states = getattr(outputs, "hidden_states")
            stacked_hidden_states = torch.stack(list(hidden_states), dim=0)
            normalized_weights = torch.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
            sequence_output = torch.sum(normalized_weights * stacked_hidden_states, dim=0)
        else:
            sequence_output = getattr(outputs, "last_hidden_state")

        if attention_mask is None:
            return sequence_output.mean(dim=1)

        if attention_mask.shape[1] != sequence_output.shape[1]:
            if not hasattr(self.backbone, "_get_feature_vector_attention_mask"):
                raise RuntimeError(
                    "Attention-mask length does not match hidden-state sequence length "
                    f"({attention_mask.shape[1]} vs {sequence_output.shape[1]}), and the "
                  "backbone does not expose _get_feature_vector_attention_mask()."
                )

            attention_mask = self.backbone._get_feature_vector_attention_mask(
                sequence_output.shape[1],
                attention_mask,
            )

        mask = attention_mask.unsqueeze(-1).to(sequence_output.dtype)
        masked_sum = torch.sum(sequence_output * mask, dim=1)
        mask_sum = torch.clamp(mask.sum(dim=1), min=1.0)
        return masked_sum / mask_sum

    def forward(self, input_values: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        backbone_kwargs = {
            "input_values": input_values,
            "attention_mask": attention_mask,
            "output_hidden_states": self.weighted_layer_pooling,
        }
        if self.freeze_backbone:
            with torch.no_grad():
                outputs = self.backbone(**backbone_kwargs)
        else:
            outputs = self.backbone(**backbone_kwargs)
        pooled = self._pool_hidden_states(outputs, attention_mask=attention_mask)
        return self.projection(pooled)


## Training, Retrieval, And Artifacts

The fixed execution order is:
1. dataset and model smoke tests
2. CNN baseline
3. hybrid transformer with baseline augmentation
4. hybrid transformer with extended augmentation
5. hybrid transformer with extended augmentation and a larger embedding
6. frozen MERT baseline

Exact retrieval uses `faiss.IndexFlatIP` over L2-normalized embeddings stored as `float32`.

### Training, retrieval, and plotting helpers

In [ ]:
def serialize_jsonable(value: object) -> object:
    """Recursively convert dataclasses and paths into JSON-safe values."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, list):
        return [serialize_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): serialize_jsonable(item) for key, item in value.items()}
    if hasattr(value, "__dataclass_fields__"):
        return {field_name: serialize_jsonable(getattr(value, field_name)) for field_name in value.__dataclass_fields__.keys()}
    return value


def make_run_directory(config: ExperimentConfig, run_config: ModelRunConfig) -> Path:
    """Create the output directory for a single run."""
    run_dir = config.paths.output_root / run_config.run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def clear_memory() -> None:
    """Release cached Python and CUDA memory between runs."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def build_model(config: ExperimentConfig, run_config: ModelRunConfig) -> tuple[nn.Module, InputMode, int, Callable[[list[dict[str, object]]], dict[str, torch.Tensor]] | None]:
    """Instantiate a model and return its input mode and optional collate function."""
    if run_config.model_name == "cnn":
        model = CompactSpectrogramCNN(run_config.embed_dim, config.projection_hidden_dim)
        return model, "spectrogram", config.sample_rate, None

    if run_config.model_name == "hybrid_transformer":
        model = HybridSpectrogramTransformer(config, run_config.embed_dim)
        return model, "spectrogram", config.sample_rate, None

    model = FrozenMertFingerprinter("m-a-p/MERT-v1-95M", run_config.embed_dim, config.projection_hidden_dim, run_config.freeze_backbone, False)
    return model, "mert_waveform", model.sample_rate, build_mert_pair_collate_fn(model.feature_extractor)


def build_retrieval_collate_fn(model: nn.Module, input_mode: InputMode) -> Callable[[list[dict[str, object]]], dict[str, torch.Tensor]]:
    """Return the retrieval collate function for a model family."""
    if input_mode == "spectrogram":
        return collate_retrieval_batch
    if not isinstance(model, FrozenMertFingerprinter):
        raise TypeError("MERT retrieval collate requires a FrozenMertFingerprinter instance.")
    return build_mert_retrieval_collate_fn(model.feature_extractor)


def move_batch_to_device(batch: dict[str, torch.Tensor], device: torch.device) -> dict[str, torch.Tensor]:
    """Move all tensor values in a batch dictionary to the requested device."""
    return {key: value.to(device) if isinstance(value, torch.Tensor) else value for key, value in batch.items()}


def forward_pair_batch(model: nn.Module, batch: dict[str, torch.Tensor], input_mode: InputMode, device_type: str, amp_enabled: bool) -> tuple[torch.Tensor, torch.Tensor]:
    """Compute anchor and positive embeddings for one training batch."""
    with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
        if input_mode == "spectrogram":
            anchor_embeddings = model(batch["anchor"])
            positive_embeddings = model(batch["positive"])
        else:
            if not isinstance(model, FrozenMertFingerprinter):
                raise TypeError("Waveform batches require FrozenMertFingerprinter.")
            anchor_embeddings = model(batch["anchor_input_values"], batch["anchor_attention_mask"])
            positive_embeddings = model(batch["positive_input_values"], batch["positive_attention_mask"])
    return anchor_embeddings, positive_embeddings


def encode_retrieval_batch(model: nn.Module, batch: dict[str, torch.Tensor], input_mode: InputMode, device: torch.device, device_type: str, amp_enabled: bool) -> torch.Tensor:
    """Encode one retrieval batch into embeddings."""
    with torch.no_grad():
        with torch.amp.autocast(device_type=device_type, enabled=amp_enabled):
            if input_mode == "spectrogram":
                return model(batch["inputs"].to(device))
            if not isinstance(model, FrozenMertFingerprinter):
                raise TypeError("Waveform retrieval batches require FrozenMertFingerprinter.")
            return model(batch["inputs"].to(device), batch["attention_mask"].to(device))


def train_one_epoch(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], optimizer: torch.optim.Optimizer, loss_fn: NTXentLoss, device: torch.device, input_mode: InputMode, config: ExperimentConfig) -> float:
    """Train a model for one epoch and return the mean loss."""
    model.train()
    if isinstance(model, FrozenMertFingerprinter) and model.freeze_backbone:
        model.backbone.eval()
    scaler = torch.amp.GradScaler(config.device_type, enabled=config.amp_enabled)
    epoch_losses: list[float] = []

    for batch in tqdm(loader, desc="train", leave=False):
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        if not torch.isfinite(loss):
            raise FloatingPointError("Encountered a non-finite training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.gradient_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(float(loss.detach().cpu().item()))

    return float(np.mean(epoch_losses))


def validate_epoch(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], loss_fn: NTXentLoss, device: torch.device, input_mode: InputMode, config: ExperimentConfig) -> float:
    """Evaluate mean contrastive loss on the validation loader."""
    model.eval()
    losses: list[float] = []
    for batch in tqdm(loader, desc="validate", leave=False):
        batch = move_batch_to_device(batch, device)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, config.device_type, config.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        losses.append(float(loss.detach().cpu().item()))
    return float(np.mean(losses))


### Embedding extraction, retrieval metrics, and artifact writers

In [ ]:
def extract_embeddings(model: nn.Module, loader: DataLoader[dict[str, torch.Tensor]], input_mode: InputMode, device: torch.device, config: ExperimentConfig) -> tuple[np.ndarray, np.ndarray]:
    """Encode a loader into a NumPy embedding matrix and aligned track IDs."""
    model.eval()
    embedding_chunks: list[np.ndarray] = []
    track_id_chunks: list[np.ndarray] = []
    for batch in tqdm(loader, desc="embed", leave=False):
        track_ids = batch["track_id"].detach().cpu().numpy().astype(np.int64)
        encoded = encode_retrieval_batch(model, batch, input_mode, device, config.device_type, config.amp_enabled)
        embedding_chunks.append(encoded.detach().cpu().numpy().astype(np.float32))
        track_id_chunks.append(track_ids)
    return np.concatenate(embedding_chunks, axis=0), np.concatenate(track_id_chunks, axis=0)


def build_faiss_index(reference_embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """Build an exact FAISS index over L2-normalized float32 embeddings."""
    if reference_embeddings.dtype != np.float32:
        reference_embeddings = reference_embeddings.astype(np.float32)
    faiss.normalize_L2(reference_embeddings)
    index = faiss.IndexFlatIP(reference_embeddings.shape[1])
    index.add(reference_embeddings)
    return index


def compute_rank_metrics(ranked_reference_ids: np.ndarray, query_track_ids: np.ndarray, top_k: tuple[int, ...]) -> dict[str, float]:
    """Compute Top-k accuracy and MRR from ranked reference IDs."""
    metrics: dict[str, float] = {}
    reciprocal_ranks: list[float] = []
    for query_index, query_track_id in enumerate(query_track_ids):
        ranked_ids = ranked_reference_ids[query_index]
        match_indices = np.where(ranked_ids == query_track_id)[0]
        rank = int(match_indices[0]) + 1 if match_indices.size else ranked_ids.shape[0] + 1
        reciprocal_ranks.append(1.0 / rank)
        for k in top_k:
            metrics.setdefault(f"top{k}", 0.0)
            metrics[f"top{k}"] += 1.0 if rank <= k else 0.0

    query_count = float(len(query_track_ids))
    for k in top_k:
        metrics[f"top{k}"] /= query_count
    metrics["mrr"] = float(np.mean(reciprocal_ranks))
    return metrics


def evaluate_retrieval(model: nn.Module, reference_loader: DataLoader[dict[str, torch.Tensor]], query_loaders: dict[RetrievalConditionName, DataLoader[dict[str, torch.Tensor]]], input_mode: InputMode, device: torch.device, config: ExperimentConfig) -> tuple[dict[str, float], pd.DataFrame]:
    """Build a FAISS index and evaluate clean plus degraded retrieval metrics."""
    reference_embeddings, reference_track_ids = extract_embeddings(model, reference_loader, input_mode, device, config)
    index = build_faiss_index(reference_embeddings.copy())
    results: dict[str, float] = {}
    degradation_rows: list[dict[str, object]] = []

    for condition, query_loader in query_loaders.items():
        query_embeddings, query_track_ids = extract_embeddings(model, query_loader, input_mode, device, config)
        faiss.normalize_L2(query_embeddings)
        _, neighbor_indices = index.search(query_embeddings.astype(np.float32), max(config.top_k))
        ranked_reference_ids = reference_track_ids[neighbor_indices]
        condition_metrics = compute_rank_metrics(ranked_reference_ids, query_track_ids, config.top_k)
        degradation_rows.append(
            {
                "condition": condition,
                "top1": condition_metrics["top1"],
                "top5": condition_metrics["top5"],
                "top10": condition_metrics["top10"],
                "mrr": condition_metrics["mrr"],
            }
        )
        if condition == "clean":
            results["clean_top1"] = condition_metrics["top1"]
            results["clean_top5"] = condition_metrics["top5"]
            results["clean_top10"] = condition_metrics["top10"]
            results["clean_mrr"] = condition_metrics["mrr"]
        else:
            results[f"{condition}_top1"] = condition_metrics["top1"]

    return results, pd.DataFrame(degradation_rows)


def save_json(filepath: Path, payload: object) -> None:
    """Write a JSON artifact with stable indentation."""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with filepath.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
        handle.write("\n")


def plot_training_history(history: list[dict[str, float]], output_path: Path, run_title: str) -> None:
    """Save a compact loss-curve plot for one experiment run."""
    epochs = [int(item["epoch"]) for item in history]
    train_losses = [float(item["train_loss"]) for item in history]
    val_losses = [float(item["val_loss"]) for item in history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, train_losses, marker="o", label="Train loss")
    ax.plot(epochs, val_losses, marker="s", label="Validation loss")
    ax.set_title(f"Loss curves | {run_title}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent loss")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_retrieval_bars(degradation_df: pd.DataFrame, output_path: Path, run_title: str) -> None:
    """Save a degradation-specific Top-1 bar chart for one experiment run."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(degradation_df["condition"], degradation_df["top1"], color="steelblue")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Top-1 accuracy")
    ax.set_title(f"Retrieval robustness | {run_title}")
    ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


## Additions: Policies, Query Regimes, Multi-Window Indexing, And Caches

The next code cell is where this notebook stops being Notebook 2 with a few extra plots.

It adds:
- policy-driven augmentation beyond `baseline` and `extended`
- richer retrieval conditions and shorter/off-center query regimes
- multi-window reference manifests for realistic indexing
- cache-aware datasets and metadata-rich loaders


### Augmentation policies, query regimes, manifests, and retrieval datasets

In [ ]:
@dataclass(frozen=True)
class AugmentationTransformSpec:
    """A single augmentation transform and its parameter sampling policy."""

    name: str
    probability: float
    parameters: dict[str, float]
    order: int = 0


@dataclass(frozen=True)
class AugmentationPolicySpec:
    """A complete augmentation policy for contrastive positive generation."""

    name: str
    description: str
    selection_strategy: str
    transforms: tuple[AugmentationTransformSpec, ...]
    max_transforms_per_sample: int
    enabled: bool = True
    curriculum_transition_epoch: int | None = None


@dataclass(frozen=True)
class QueryRegimeSpec:
    """How to sample retrieval queries for one evaluation regime."""

    name: QueryRegimeName
    description: str
    lengths_seconds: tuple[float, ...]
    centered: bool
    random_offset: bool
    conditions: tuple[RetrievalConditionName, ...]
    group_size: int
    enabled: bool = True
    apply_fade_envelope: bool = False


@dataclass(frozen=True)
class WindowingSpec:
    """Reference-window construction strategy for indexing."""

    name: WindowingSpecName
    description: str
    window_count: int
    segment_seconds: float
    centered_only: bool
    enabled: bool = True


@dataclass(frozen=True)
class IndexSpec:
    """FAISS index configuration."""

    name: str
    kind: IndexKind
    description: str
    nprobe: int | None = None
    enabled: bool = True
    max_nlist: int = 64


def validate_probability(name: str, value: float) -> None:
    """Validate that a probability lies inside the closed interval [0, 1]."""
    if value < 0.0 or value > 1.0:
        raise ValueError(f"{name} must lie in [0, 1], received {value}.")


def apply_fade_envelope(waveform: np.ndarray, sample_rate: int, fade_seconds: float) -> np.ndarray:
    """Apply a symmetric fade-in/fade-out envelope."""
    validate_non_empty_waveform(waveform)
    fade_samples = min(int(max(0.0, fade_seconds) * sample_rate), waveform.shape[0] // 2)
    if fade_samples <= 1:
        return waveform.astype(np.float32)
    envelope = np.ones_like(waveform, dtype=np.float32)
    fade_curve = np.linspace(0.0, 1.0, fade_samples, dtype=np.float32)
    envelope[:fade_samples] = fade_curve
    envelope[-fade_samples:] = fade_curve[::-1]
    return (waveform * envelope).astype(np.float32)


def build_augmentation_policy_registry() -> dict[str, AugmentationPolicySpec]:
    """Construct the augmentation-policy registry."""
    registry = {
        "baseline": AugmentationPolicySpec(
            "baseline",
            "Exact Notebook 2 baseline policy.",
            "notebook2_exact",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -6.0, "max_db": 6.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 10.0, "max_snr_db": 25.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.92, "max_rate": 1.08}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -2.0, "max_steps": 2.0}, 3),
            ),
            4,
        ),
        "extended_existing": AugmentationPolicySpec(
            "extended_existing",
            "Exact Notebook 2 extended policy.",
            "notebook2_exact",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -6.0, "max_db": 6.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 10.0, "max_snr_db": 25.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.92, "max_rate": 1.08}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -2.0, "max_steps": 2.0}, 3),
                AugmentationTransformSpec("lowpass", 1.0, {"min_cutoff_hz": 2500.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 1.0, {"min_cutoff_hz": 120.0, "max_cutoff_hz": 900.0}, 5),
            ),
            6,
        ),
        "filters_only": AugmentationPolicySpec(
            "filters_only",
            "Only low-pass or high-pass filtering.",
            "up_to_k",
            (
                AugmentationTransformSpec("lowpass", 0.55, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5200.0}, 0),
                AugmentationTransformSpec("highpass", 0.55, {"min_cutoff_hz": 160.0, "max_cutoff_hz": 720.0}, 1),
            ),
            1,
        ),
        "one_of_k": AugmentationPolicySpec(
            "one_of_k",
            "Apply exactly one augmentation from a broad candidate set.",
            "exactly_one",
            (
                AugmentationTransformSpec("gain", 1.0, {"min_db": -5.0, "max_db": 5.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 1.0, {"min_snr_db": 12.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 1.0, {"min_rate": 0.94, "max_rate": 1.06}, 2),
                AugmentationTransformSpec("pitch_shift", 1.0, {"min_steps": -1.5, "max_steps": 1.5}, 3),
                AugmentationTransformSpec("lowpass", 1.0, {"min_cutoff_hz": 2600.0, "max_cutoff_hz": 4800.0}, 4),
                AugmentationTransformSpec("highpass", 1.0, {"min_cutoff_hz": 140.0, "max_cutoff_hz": 700.0}, 5),
            ),
            1,
        ),
        "two_of_k": AugmentationPolicySpec(
            "two_of_k",
            "Apply at most two augmentations from a broad candidate set.",
            "up_to_k",
            (
                AugmentationTransformSpec("gain", 0.45, {"min_db": -5.0, "max_db": 5.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 0.60, {"min_snr_db": 12.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 0.40, {"min_rate": 0.95, "max_rate": 1.05}, 2),
                AugmentationTransformSpec("pitch_shift", 0.40, {"min_steps": -1.5, "max_steps": 1.5}, 3),
                AugmentationTransformSpec("lowpass", 0.35, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 0.35, {"min_cutoff_hz": 150.0, "max_cutoff_hz": 650.0}, 5),
            ),
            2,
            False,
        ),
        "severity_controlled": AugmentationPolicySpec(
            "severity_controlled",
            "Apply one or two milder transforms with narrower parameter ranges.",
            "up_to_k",
            (
                AugmentationTransformSpec("gain", 0.35, {"min_db": -3.0, "max_db": 3.0}, 0),
                AugmentationTransformSpec("gaussian_noise", 0.55, {"min_snr_db": 16.0, "max_snr_db": 24.0}, 1),
                AugmentationTransformSpec("time_stretch", 0.35, {"min_rate": 0.97, "max_rate": 1.03}, 2),
                AugmentationTransformSpec("pitch_shift", 0.35, {"min_steps": -1.0, "max_steps": 1.0}, 3),
                AugmentationTransformSpec("lowpass", 0.25, {"min_cutoff_hz": 3200.0, "max_cutoff_hz": 4600.0}, 4),
                AugmentationTransformSpec("highpass", 0.25, {"min_cutoff_hz": 180.0, "max_cutoff_hz": 520.0}, 5),
            ),
            2,
        ),
        "curriculum_filters_late": AugmentationPolicySpec(
            "curriculum_filters_late",
            "Start with baseline augmentation, then introduce filters later.",
            "curriculum",
            (
                AugmentationTransformSpec("lowpass", 0.45, {"min_cutoff_hz": 2800.0, "max_cutoff_hz": 5000.0}, 4),
                AugmentationTransformSpec("highpass", 0.45, {"min_cutoff_hz": 150.0, "max_cutoff_hz": 650.0}, 5),
            ),
            2,
            False,
            2,
        ),
    }
    for policy in registry.values():
        if policy.max_transforms_per_sample <= 0:
            raise ValueError(f"Policy {policy.name} must allow at least one transform.")
        for transform in policy.transforms:
            validate_probability(f"{policy.name}:{transform.name}", float(transform.probability))
    return registry


def sample_transform_parameters(transform: AugmentationTransformSpec, rng: np.random.Generator) -> dict[str, float]:
    """Sample concrete transform parameters from a transform specification."""
    sampled: dict[str, float] = {}
    if "min_db" in transform.parameters and "max_db" in transform.parameters:
        sampled["gain_db"] = float(rng.uniform(transform.parameters["min_db"], transform.parameters["max_db"]))
    if "min_snr_db" in transform.parameters and "max_snr_db" in transform.parameters:
        sampled["snr_db"] = float(rng.uniform(transform.parameters["min_snr_db"], transform.parameters["max_snr_db"]))
    if "min_rate" in transform.parameters and "max_rate" in transform.parameters:
        sampled["rate"] = float(rng.uniform(transform.parameters["min_rate"], transform.parameters["max_rate"]))
    if "min_steps" in transform.parameters and "max_steps" in transform.parameters:
        sampled["n_steps"] = float(rng.uniform(transform.parameters["min_steps"], transform.parameters["max_steps"]))
    if "min_cutoff_hz" in transform.parameters and "max_cutoff_hz" in transform.parameters:
        sampled["cutoff_hz"] = float(rng.uniform(transform.parameters["min_cutoff_hz"], transform.parameters["max_cutoff_hz"]))
    return sampled


def apply_transform_with_metadata(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    transform: AugmentationTransformSpec,
    rng: np.random.Generator,
) -> tuple[np.ndarray, dict[str, object]]:
    """Apply one transform and return the transformed waveform plus metadata."""
    sampled = sample_transform_parameters(transform, rng)
    transformed = waveform.copy()
    if transform.name == "gain":
        transformed = apply_random_gain(transformed, rng=rng, min_db=sampled["gain_db"], max_db=sampled["gain_db"])
    elif transform.name == "gaussian_noise":
        transformed = add_gaussian_noise(transformed, rng=rng, snr_db=sampled["snr_db"])
    elif transform.name == "time_stretch":
        transformed = apply_time_stretch(transformed, rate=sampled["rate"])
    elif transform.name == "pitch_shift":
        transformed = apply_pitch_shift(transformed, sample_rate=sample_rate, n_steps=sampled["n_steps"])
    elif transform.name == "lowpass":
        transformed = apply_lowpass_filter(transformed, sample_rate=sample_rate, cutoff_hz=sampled["cutoff_hz"])
    elif transform.name == "highpass":
        transformed = apply_highpass_filter(transformed, sample_rate=sample_rate, cutoff_hz=sampled["cutoff_hz"])
    else:
        raise ValueError(f"Unsupported transform name: {transform.name}")
    transformed = trim_or_pad_waveform(transformed, target_length)
    return transformed.astype(np.float32), {"transform_name": transform.name, "parameters": sampled}


def select_policy_transforms(
    policy: AugmentationPolicySpec,
    registry: dict[str, AugmentationPolicySpec],
    rng: np.random.Generator,
    current_epoch: int | None,
) -> tuple[AugmentationTransformSpec, ...]:
    """Resolve which transforms to apply for one policy invocation."""
    ordered_transforms = tuple(sorted(policy.transforms, key=lambda item: item.order))
    if policy.selection_strategy == "notebook2_exact":
        return ordered_transforms
    if policy.selection_strategy == "curriculum":
        transition_epoch = int(policy.curriculum_transition_epoch or 0)
        if current_epoch is None or current_epoch < transition_epoch:
            return tuple(sorted(registry["baseline"].transforms, key=lambda item: item.order))
        late_candidates = list(tuple(sorted(registry["baseline"].transforms, key=lambda item: item.order)) + ordered_transforms)
        selected = [candidate for candidate in late_candidates if rng.random() <= float(candidate.probability)]
        if not selected:
            selected = [late_candidates[int(rng.integers(0, len(late_candidates)))]]
        return tuple(selected[: policy.max_transforms_per_sample])
    if policy.selection_strategy == "exactly_one":
        weights = np.asarray([max(1e-6, float(item.probability)) for item in ordered_transforms], dtype=np.float64)
        weights = weights / weights.sum()
        selected_index = int(rng.choice(np.arange(len(ordered_transforms)), p=weights))
        return (ordered_transforms[selected_index],)
    selected = [item for item in ordered_transforms if rng.random() <= float(item.probability)]
    if not selected:
        weights = np.asarray([max(1e-6, float(item.probability)) for item in ordered_transforms], dtype=np.float64)
        weights = weights / weights.sum()
        selected = [ordered_transforms[int(rng.choice(np.arange(len(ordered_transforms)), p=weights))]]
    if len(selected) > policy.max_transforms_per_sample:
        selected_indices = rng.choice(np.arange(len(selected)), size=policy.max_transforms_per_sample, replace=False)
        selected = [selected[int(index)] for index in np.sort(selected_indices)]
    return tuple(sorted(selected, key=lambda item: item.order))


def apply_augmented_policy(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    profile: str,
    rng: np.random.Generator,
    current_epoch: int | None = None,
) -> tuple[np.ndarray, list[dict[str, object]]]:
    """Apply one named augmentation policy and return waveform plus metadata."""
    if profile in {"baseline", "extended"}:
        legacy_waveform = apply_augmentation_profile(
            waveform=waveform,
            sample_rate=sample_rate,
            target_length=target_length,
            profile="baseline" if profile == "baseline" else "extended",
            rng=rng,
        )
        return legacy_waveform.astype(np.float32), [{"transform_name": profile, "parameters": {"legacy_path": True}}]
    policy = AUGMENTATION_POLICY_REGISTRY[profile]
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    metadata_rows: list[dict[str, object]] = []
    for transform in select_policy_transforms(policy, AUGMENTATION_POLICY_REGISTRY, rng, current_epoch):
        segment, metadata = apply_transform_with_metadata(segment, sample_rate, target_length, transform, rng)
        metadata_rows.append(metadata)
    segment = normalize_waveform(segment)
    if not np.isfinite(segment).all():
        raise FloatingPointError(f"Policy {profile} produced non-finite waveform values.")
    return segment.astype(np.float32), metadata_rows


### Query-regime registry, windowing registry, manifest builders, and dataset classes

In [ ]:
class ContrastivePairDataset(Dataset[dict[str, object]]):
    """Yield anchor and positive pairs for contrastive training."""

    def __init__(
        self,
        audio_dir: Path,
        track_ids: list[int],
        segment_length: int,
        sample_rate: int,
        input_mode: InputMode,
        config: ExperimentConfig,
        augmentation_profile: str,
    ) -> None:
        self.audio_dir = audio_dir
        self.track_ids = track_ids
        self.segment_length = segment_length
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config
        self.augmentation_profile = augmentation_profile
        self.current_epoch = 0

    def __len__(self) -> int:
        return len(self.track_ids)

    def __getitem__(self, index: int) -> dict[str, object]:
        track_id = int(self.track_ids[index])
        seed = int(torch.randint(low=0, high=2**31 - 1, size=(1,)).item())
        rng = np.random.default_rng(seed)
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, track_id), target_sr=self.sample_rate)
        segment, start_index = extract_random_segment(waveform, self.segment_length, rng=rng)
        segment = normalize_waveform(trim_or_pad_waveform(segment, self.segment_length))
        positive_waveform, _ = apply_augmented_policy(segment.copy(), self.sample_rate, self.segment_length, self.augmentation_profile, rng, self.current_epoch)
        if self.input_mode == "spectrogram":
            anchor_value: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
            positive_value: torch.Tensor = spectrogram_tensor_from_waveform(positive_waveform, self.config)
        else:
            anchor_value = torch.from_numpy(segment).float()
            positive_value = torch.from_numpy(positive_waveform).float()
        return {"anchor": anchor_value, "positive": positive_value, "track_id": track_id, "anchor_start_time": float(start_index / self.sample_rate)}

def apply_query_condition_v2(
    waveform: np.ndarray,
    sample_rate: int,
    target_length: int,
    condition: RetrievalConditionName,
    rng: np.random.Generator,
    apply_fade: bool = False,
) -> tuple[np.ndarray, list[dict[str, object]]]:
    """Apply one retrieval degradation condition and return transformed audio plus metadata."""
    segment = trim_or_pad_waveform(normalize_waveform(waveform), target_length)
    metadata: list[dict[str, object]] = []

    def record(name: str, parameters: dict[str, object]) -> None:
        metadata.append({"transform_name": name, "parameters": serialize_jsonable(parameters)})

    if condition == "clean":
        output = segment
    elif condition in {"noise", "pitch", "time", "lowpass", "highpass"}:
        output = apply_retrieval_degradation(segment, sample_rate, target_length, condition if condition != "time" else "time", rng)
        record(condition, {"legacy_path": True})
    elif condition == "combined_moderate":
        output = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-3.0, max_db=3.0), target_length)
        record("gain", {"policy": "moderate"})
        snr_db = float(rng.uniform(12.0, 18.0))
        output = trim_or_pad_waveform(add_gaussian_noise(output, rng=rng, snr_db=snr_db), target_length)
        record("gaussian_noise", {"snr_db": snr_db})
        secondary_conditions = ["pitch", "time", "lowpass", "highpass"]
        rng.shuffle(secondary_conditions)
        for secondary_condition in secondary_conditions[:2]:
            if secondary_condition == "pitch":
                n_steps = float(rng.uniform(-1.0, 1.0))
                output = trim_or_pad_waveform(apply_pitch_shift(output, sample_rate=sample_rate, n_steps=n_steps), target_length)
                record("pitch_shift", {"n_steps": n_steps})
            elif secondary_condition == "time":
                rate = float(rng.uniform(0.97, 1.03))
                output = trim_or_pad_waveform(apply_time_stretch(output, rate=rate), target_length)
                record("time_stretch", {"rate": rate})
            elif secondary_condition == "lowpass":
                cutoff_hz = float(rng.uniform(3200.0, 4500.0))
                output = trim_or_pad_waveform(apply_lowpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
                record("lowpass", {"cutoff_hz": cutoff_hz})
            elif secondary_condition == "highpass":
                cutoff_hz = float(rng.uniform(180.0, 480.0))
                output = trim_or_pad_waveform(apply_highpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
                record("highpass", {"cutoff_hz": cutoff_hz})
    elif condition == "realistic_hard":
        output = trim_or_pad_waveform(apply_random_gain(segment, rng=rng, min_db=-4.0, max_db=4.0), target_length)
        record("gain", {"policy": "realistic_hard"})
        snr_db = float(rng.uniform(6.0, 12.0))
        output = trim_or_pad_waveform(add_gaussian_noise(output, rng=rng, snr_db=snr_db), target_length)
        record("gaussian_noise", {"snr_db": snr_db})
        if rng.random() < 0.5:
            cutoff_hz = float(rng.uniform(2600.0, 4200.0))
            output = trim_or_pad_waveform(apply_lowpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
            record("lowpass", {"cutoff_hz": cutoff_hz})
        else:
            cutoff_hz = float(rng.uniform(220.0, 620.0))
            output = trim_or_pad_waveform(apply_highpass_filter(output, sample_rate=sample_rate, cutoff_hz=cutoff_hz), target_length)
            record("highpass", {"cutoff_hz": cutoff_hz})
        rate = float(rng.uniform(0.93, 1.07))
        output = trim_or_pad_waveform(apply_time_stretch(output, rate=rate), target_length)
        record("time_stretch", {"rate": rate})
    else:
        raise ValueError(f"Unsupported retrieval condition: {condition}")

    if apply_fade:
        fade_seconds = float(rng.uniform(0.04, 0.12))
        output = apply_fade_envelope(output, sample_rate=sample_rate, fade_seconds=fade_seconds)
        record("fade_envelope", {"fade_seconds": fade_seconds})

    output = normalize_waveform(trim_or_pad_waveform(output, target_length))
    if not np.isfinite(output).all():
        raise FloatingPointError(f"Query condition {condition} produced non-finite waveform values.")
    return output.astype(np.float32), metadata


def choose_random_offcenter_start(total_samples: int, segment_length: int, rng: np.random.Generator) -> int:
    """Sample an off-center segment start while avoiding the exact centered crop when possible."""
    if total_samples <= segment_length:
        return 0
    max_start = total_samples - segment_length
    center_start = max_start // 2
    exclusion_radius = max(1, int(0.1 * max_start))
    for _ in range(32):
        candidate = int(rng.integers(0, max_start + 1))
        if abs(candidate - center_start) >= exclusion_radius:
            return candidate
    return 0 if center_start > max_start / 2 else max_start


def build_even_window_starts(total_samples: int, segment_length: int, window_count: int) -> list[int]:
    """Create evenly spaced segment starts for multi-window indexing."""
    if window_count <= 0:
        raise ValueError(f"window_count must be positive, received {window_count}.")
    if total_samples <= segment_length:
        return [0]
    max_start = total_samples - segment_length
    starts = np.linspace(0, max_start, num=window_count, dtype=np.int64)
    return sorted({int(start_value) for start_value in starts})


def build_query_regime_registry(config: ExperimentConfig) -> dict[QueryRegimeName, QueryRegimeSpec]:
    """Construct the retrieval-query regime registry."""
    return {
        "clean_current": QueryRegimeSpec("clean_current", "Notebook 2 continuity baseline.", (config.segment_seconds,), True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_centered": QueryRegimeSpec("short_centered", "Short centered queries.", config.short_query_lengths_seconds, True, False, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "short_offcenter": QueryRegimeSpec("short_offcenter", "Short off-center queries.", config.short_query_lengths_seconds, False, True, ("clean", "noise", "pitch", "time", "lowpass", "highpass"), 1),
        "combined_moderate": QueryRegimeSpec("combined_moderate", "Off-center queries with moderate combined degradations.", config.short_query_lengths_seconds, False, True, ("combined_moderate",), 1),
        "multi_segment_same_track": QueryRegimeSpec("multi_segment_same_track", "Aggregate several short segments from one track.", (1.0, 2.0), False, True, ("clean", "combined_moderate"), config.multi_segment_group_size),
        "realistic_hard": QueryRegimeSpec("realistic_hard", "Harder off-center queries with fade envelopes.", config.short_query_lengths_seconds, False, True, ("realistic_hard",), 1, False, True),
    }


def build_windowing_registry(config: ExperimentConfig) -> dict[WindowingSpecName, WindowingSpec]:
    """Construct the reference-window indexing registry."""
    return {
        "single_center": WindowingSpec("single_center", "Notebook 2 single centered reference segment.", 1, config.segment_seconds, True),
        "multi3_even": WindowingSpec("multi3_even", "Three evenly spaced reference windows.", 3, config.segment_seconds, False),
        "multi5_even": WindowingSpec("multi5_even", "Five evenly spaced reference windows.", 5, config.segment_seconds, False, False),
    }


class ReferenceManifestDataset(Dataset[dict[str, object]]):
    """Yield deterministic reference windows from a precomputed manifest."""

    def __init__(self, audio_dir: Path, manifest_df: pd.DataFrame, sample_rate: int, input_mode: InputMode, config: ExperimentConfig) -> None:
        self.audio_dir = audio_dir
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return int(len(self.manifest_df))

    def __getitem__(self, index: int) -> dict[str, object]:
        row = self.manifest_df.iloc[int(index)]
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, int(row["track_id"])), target_sr=self.sample_rate)
        segment_length = int(float(row["segment_length_seconds"]) * self.sample_rate)
        segment = extract_segment_by_start(waveform, int(row["start_sample"]), segment_length)
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(segment, self.config)
        else:
            model_input = torch.from_numpy(segment).float()
        return {"inputs": model_input, "track_id": int(row["track_id"]), "metadata_row": row.to_dict()}


class QueryManifestDataset(Dataset[dict[str, object]]):
    """Yield deterministic retrieval queries from a precomputed manifest."""

    def __init__(self, audio_dir: Path, manifest_df: pd.DataFrame, sample_rate: int, input_mode: InputMode, config: ExperimentConfig) -> None:
        self.audio_dir = audio_dir
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.sample_rate = sample_rate
        self.input_mode = input_mode
        self.config = config

    def __len__(self) -> int:
        return int(len(self.manifest_df))

    def __getitem__(self, index: int) -> dict[str, object]:
        row = self.manifest_df.iloc[int(index)]
        waveform, _ = load_audio_file(get_audio_path(self.audio_dir, int(row["track_id"])), target_sr=self.sample_rate)
        segment_length = int(float(row["query_length_seconds"]) * self.sample_rate)
        segment = extract_segment_by_start(waveform, int(row["start_sample"]), segment_length)
        rng_seed = int(self.config.seed + int(row["track_id"]) * 17 + int(index) * 13)
        rng = np.random.default_rng(rng_seed)
        degraded_segment, degradation_metadata = apply_query_condition_v2(segment, self.sample_rate, segment_length, str(row["condition_name"]), rng, bool(row["apply_fade_envelope"]))
        if self.input_mode == "spectrogram":
            model_input: torch.Tensor = spectrogram_tensor_from_waveform(degraded_segment, self.config)
        else:
            model_input = torch.from_numpy(degraded_segment).float()
        metadata_row = row.to_dict()
        metadata_row["degradation_metadata"] = degradation_metadata
        return {"inputs": model_input, "track_id": int(row["track_id"]), "metadata_row": metadata_row}


def collate_retrieval_manifest_batch(batch: list[dict[str, object]]) -> RetrievalBatch:
    """Stack retrieval samples while preserving per-row metadata."""
    inputs = torch.stack([sample["inputs"] for sample in batch])
    track_ids = torch.tensor([int(sample["track_id"]) for sample in batch], dtype=torch.long)
    metadata_rows = [dict(sample["metadata_row"]) for sample in batch]
    return {"inputs": inputs, "track_id": track_ids, "metadata_rows": metadata_rows}


AUGMENTATION_POLICY_REGISTRY = build_augmentation_policy_registry()
QUERY_REGIME_REGISTRY = build_query_regime_registry(CONFIG)
WINDOWING_REGISTRY = build_windowing_registry(CONFIG)

AUGMENTATION_POLICY_SUMMARY = pd.DataFrame(
    [
        {
            "policy_name": policy.name,
            "selection_strategy": policy.selection_strategy,
            "max_transforms_per_sample": policy.max_transforms_per_sample,
            "enabled": policy.enabled,
            "transforms": ", ".join(transform.name for transform in policy.transforms),
        }
        for policy in AUGMENTATION_POLICY_REGISTRY.values()
    ]
)
QUERY_REGIME_SUMMARY = pd.DataFrame([serialize_jsonable(regime) for regime in QUERY_REGIME_REGISTRY.values()])
WINDOWING_SUMMARY = pd.DataFrame([serialize_jsonable(spec) for spec in WINDOWING_REGISTRY.values()])
display(AUGMENTATION_POLICY_SUMMARY)
display(QUERY_REGIME_SUMMARY)
display(WINDOWING_SUMMARY)


## Experiment Orchestration

This section resolves the active training runs, query regimes, windowing specs, and FAISS indexes, then drives:
- smoke tests
- new augmentation-policy training runs
- full retrieval evaluation
- final exports


### Loaders, manifests, FAISS sweep, training orchestration, and final synthesis helpers

In [ ]:
@dataclass(frozen=True)
class ArtifactDescriptor:
    """A resolved model artifact that can be loaded and evaluated."""

    run_id: str
    source: str
    model_name: ModelName
    augmentation_profile: str
    embed_dim: int
    checkpoint_path: Path
    freeze_backbone: bool
    notes: str = ""


def build_index_spec_registry(config: ExperimentConfig) -> dict[str, IndexSpec]:
    """Construct the bounded FAISS index sweep."""
    registry: dict[str, IndexSpec] = {"exact_ip": IndexSpec("exact_ip", "exact_ip", "Exact inner-product FAISS baseline.")}
    for nprobe_value in config.faiss_nprobe_values:
        registry[f"ivfflat_nprobe{nprobe_value}"] = IndexSpec(f"ivfflat_nprobe{nprobe_value}", "ivfflat", f"IVFFlat with nprobe={nprobe_value}.", int(nprobe_value))
        registry[f"ivfpq_nprobe{nprobe_value}"] = IndexSpec(f"ivfpq_nprobe{nprobe_value}", "ivfpq", f"IVFPQ with nprobe={nprobe_value}.", int(nprobe_value))
    return registry


def build_active_control_artifacts(artifact_df: pd.DataFrame, config: ExperimentConfig) -> list[ArtifactDescriptor]:
    """Convert discovered Notebook 2 checkpoints into active artifact descriptors."""
    available_rows = artifact_df[artifact_df["checkpoint_exists"]].copy()
    descriptors = [
        ArtifactDescriptor(
            run_id=str(row.run_id),
            source="notebook2",
            model_name=str(row.model_name),
            augmentation_profile=str(row.augmentation_profile),
            embed_dim=int(row.embed_dim),
            checkpoint_path=Path(str(row.checkpoint_path)),
            freeze_backbone=bool(row.freeze_backbone),
            notes="Notebook 2 control artifact.",
        )
        for row in available_rows.itertuples(index=False)
        if str(row.run_id) in config.active_control_run_ids
    ]
    return sorted(descriptors, key=lambda descriptor: descriptor.run_id)


def build_reference_manifest(track_ids: list[int], windowing_spec: WindowingSpec, config: ExperimentConfig, force_rebuild: bool = False) -> pd.DataFrame:
    """Build or load the cached reference manifest for one windowing strategy."""
    manifest_dir = safe_mkdir(config.paths.cache_dir / "reference_manifests")
    cache_key = hash_experiment_config({"windowing_spec": windowing_spec, "track_ids": track_ids, "sample_rate": config.sample_rate})
    manifest_path = manifest_dir / f"{windowing_spec.name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild:
        return pd.read_csv(manifest_path)

    rows: list[dict[str, object]] = []
    target_length = int(float(windowing_spec.segment_seconds) * config.sample_rate)
    for track_id in tqdm(track_ids, desc=f"reference-manifest:{windowing_spec.name}", leave=False):
        waveform, _ = load_audio_file(get_audio_path(config.paths.audio_dir, int(track_id)), target_sr=config.sample_rate)
        total_samples = int(waveform.shape[0])
        if windowing_spec.centered_only:
            _, center_start = extract_center_segment(waveform, target_length)
            starts = [center_start]
        else:
            starts = build_even_window_starts(total_samples, target_length, int(windowing_spec.window_count))
        for window_index, start_sample in enumerate(starts):
            rows.append(
                {
                    "record_id": f"{windowing_spec.name}:{track_id}:{window_index}",
                    "track_id": int(track_id),
                    "windowing_name": windowing_spec.name,
                    "window_index": int(window_index),
                    "segment_length_seconds": float(windowing_spec.segment_seconds),
                    "start_sample": int(start_sample),
                    "end_sample": int(start_sample + target_length),
                    "start_seconds": float(start_sample / config.sample_rate),
                    "end_seconds": float((start_sample + target_length) / config.sample_rate),
                    "cache_key": cache_key,
                }
            )
    manifest_df = pd.DataFrame(rows)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


def build_query_manifest(track_ids: list[int], regime_spec: QueryRegimeSpec, config: ExperimentConfig, force_rebuild: bool = False) -> pd.DataFrame:
    """Build or load the cached query manifest for one evaluation regime."""
    manifest_dir = safe_mkdir(config.paths.cache_dir / "query_manifests")
    cache_key = hash_experiment_config({"regime_spec": regime_spec, "track_ids": track_ids, "sample_rate": config.sample_rate})
    manifest_path = manifest_dir / f"{regime_spec.name}_{cache_key}.csv"
    if manifest_path.exists() and not force_rebuild:
        return pd.read_csv(manifest_path)

    rows: list[dict[str, object]] = []
    for track_id in tqdm(track_ids, desc=f"query-manifest:{regime_spec.name}", leave=False):
        waveform, _ = load_audio_file(get_audio_path(config.paths.audio_dir, int(track_id)), target_sr=config.sample_rate)
        total_samples = int(waveform.shape[0])
        for query_length_seconds in regime_spec.lengths_seconds:
            target_length = int(float(query_length_seconds) * config.sample_rate)
            for condition_name in regime_spec.conditions:
                if regime_spec.group_size == 1:
                    rng = np.random.default_rng(int(config.seed + int(track_id) * 31 + int(float(query_length_seconds) * 1000)))
                    if regime_spec.centered:
                        _, start_sample = extract_center_segment(waveform, target_length)
                    elif regime_spec.random_offset:
                        start_sample = choose_random_offcenter_start(total_samples, target_length, rng)
                    else:
                        start_sample = 0
                    query_group_id = f"{regime_spec.name}:{track_id}:{condition_name}:{query_length_seconds:.2f}"
                    rows.append(
                        {
                            "record_id": f"{query_group_id}:0",
                            "query_group_id": query_group_id,
                            "track_id": int(track_id),
                            "regime_name": regime_spec.name,
                            "condition_name": condition_name,
                            "query_length_seconds": float(query_length_seconds),
                            "group_size": 1,
                            "segment_index": 0,
                            "start_sample": int(start_sample),
                            "end_sample": int(start_sample + target_length),
                            "start_seconds": float(start_sample / config.sample_rate),
                            "end_seconds": float((start_sample + target_length) / config.sample_rate),
                            "apply_fade_envelope": bool(regime_spec.apply_fade_envelope),
                            "cache_key": cache_key,
                        }
                    )
                else:
                    rng = np.random.default_rng(int(config.seed + int(track_id) * 101 + int(float(query_length_seconds) * 1000)))
                    base_starts = build_even_window_starts(total_samples, target_length, int(regime_spec.group_size))
                    jitter_limit = max(1, int(0.05 * config.sample_rate))
                    query_group_id = f"{regime_spec.name}:{track_id}:{condition_name}:{query_length_seconds:.2f}"
                    for segment_index, base_start in enumerate(base_starts):
                        jitter = int(rng.integers(-jitter_limit, jitter_limit + 1))
                        max_start = max(0, total_samples - target_length)
                        start_sample = min(max_start, max(0, base_start + jitter))
                        rows.append(
                            {
                                "record_id": f"{query_group_id}:{segment_index}",
                                "query_group_id": query_group_id,
                                "track_id": int(track_id),
                                "regime_name": regime_spec.name,
                                "condition_name": condition_name,
                                "query_length_seconds": float(query_length_seconds),
                                "group_size": int(regime_spec.group_size),
                                "segment_index": int(segment_index),
                                "start_sample": int(start_sample),
                                "end_sample": int(start_sample + target_length),
                                "start_seconds": float(start_sample / config.sample_rate),
                                "end_seconds": float((start_sample + target_length) / config.sample_rate),
                                "apply_fade_envelope": bool(regime_spec.apply_fade_envelope),
                                "cache_key": cache_key,
                            }
                        )
    manifest_df = pd.DataFrame(rows)
    save_dataframe(manifest_path, manifest_df)
    return manifest_df


### Embedding caches, FAISS search, training execution, evaluation, and final synthesis


In [ ]:
def make_retrieval_loader_v2(dataset: Dataset[dict[str, object]], batch_size: int, collate_fn: Callable[[list[dict[str, object]]], dict[str, object]], num_workers: int) -> DataLoader[dict[str, object]]:
    """Build a deterministic retrieval loader for manifest datasets."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=bool(num_workers > 0),
        worker_init_fn=seed_worker,
        generator=build_torch_generator(CONFIG.seed),
        collate_fn=collate_fn,
    )


def extract_embeddings_with_cache_v2(model: nn.Module, loader: DataLoader[dict[str, object]], input_mode: InputMode, manifest_df: pd.DataFrame, cache_namespace: str, model_cache_key: str) -> tuple[np.ndarray, pd.DataFrame, dict[str, object]]:
    """Encode a manifest into embeddings, using the on-disk cache when available."""
    manifest_hash = dataframe_hash(manifest_df)
    cache_key = hash_experiment_config({"namespace": cache_namespace, "manifest_hash": manifest_hash, "model_cache_key": model_cache_key})
    embedding_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.npy"
    metadata_path = CONFIG.paths.embedding_cache_dir / f"{cache_key}.csv"
    if embedding_path.exists() and metadata_path.exists():
        return np.load(embedding_path).astype(np.float32), pd.read_csv(metadata_path), {"cache_key": cache_key, "cache_hit": True, "embedding_path": embedding_path, "metadata_path": metadata_path}

    model.eval()
    embedding_chunks: list[np.ndarray] = []
    metadata_rows: list[dict[str, object]] = []
    for batch in tqdm(loader, desc=f"embed:{cache_namespace}", leave=False):
        encoded = encode_retrieval_batch(model, batch, input_mode, CONFIG.device, CONFIG.device_type, CONFIG.amp_enabled)
        embedding_chunks.append(encoded.detach().cpu().numpy().astype(np.float32))
        metadata_rows.extend([dict(metadata_row) for metadata_row in batch["metadata_rows"]])
    embeddings = np.concatenate(embedding_chunks, axis=0).astype(np.float32)
    metadata_df = pd.DataFrame(metadata_rows)
    np.save(embedding_path, embeddings)
    save_dataframe(metadata_path, metadata_df)
    return embeddings, metadata_df, {"cache_key": cache_key, "cache_hit": False, "embedding_path": embedding_path, "metadata_path": metadata_path}


def resolve_index_runtime_parameters(index_spec: IndexSpec, reference_count: int, embed_dim: int) -> dict[str, int]:
    """Resolve data-dependent FAISS parameters for an index specification."""
    if index_spec.kind == "exact_ip":
        return {"nlist": 0, "nprobe": 0, "pq_m": 0}
    nlist = max(1, min(int(index_spec.max_nlist), max(1, reference_count // 32)))
    nprobe = min(int(index_spec.nprobe or 1), nlist)
    if index_spec.kind == "ivfpq":
        if embed_dim % 16 == 0:
            pq_m = 16
        elif embed_dim % 8 == 0:
            pq_m = 8
        else:
            raise ValueError(f"Embedding dimension {embed_dim} is incompatible with the planned PQ partitions.")
    else:
        pq_m = 0
    return {"nlist": int(nlist), "nprobe": int(nprobe), "pq_m": int(pq_m)}


def build_or_load_faiss_index_v2(reference_embeddings: np.ndarray, index_spec: IndexSpec, reference_cache_key: str) -> tuple[faiss.Index, dict[str, object]]:
    """Build or load a cached FAISS index for one reference embedding set."""
    index_cache_key = hash_experiment_config({"reference_cache_key": reference_cache_key, "index_name": index_spec.name, "index_kind": index_spec.kind})
    index_path = CONFIG.paths.index_dir / f"{index_cache_key}.faiss"
    metadata_path = CONFIG.paths.index_dir / f"{index_cache_key}.json"
    if index_path.exists() and metadata_path.exists():
        return faiss.read_index(str(index_path)), dict(load_json(metadata_path))

    normalized_embeddings = normalize_embeddings(reference_embeddings)
    reference_count, embed_dim = normalized_embeddings.shape
    runtime_parameters = resolve_index_runtime_parameters(index_spec, reference_count, embed_dim)
    build_started_at = time.perf_counter()
    if index_spec.kind == "exact_ip":
        index = faiss.IndexFlatIP(embed_dim)
        index.add(normalized_embeddings)
    elif index_spec.kind == "ivfflat":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFFlat stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFFlat(quantizer, embed_dim, runtime_parameters["nlist"], faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    elif index_spec.kind == "ivfpq":
        if reference_count < max(runtime_parameters["nlist"] * 8, 256):
            raise ValueError("Not enough reference windows to train IVFPQ stably.")
        quantizer = faiss.IndexFlatIP(embed_dim)
        index = faiss.IndexIVFPQ(quantizer, embed_dim, runtime_parameters["nlist"], runtime_parameters["pq_m"], 8, faiss.METRIC_INNER_PRODUCT)
        index.train(normalized_embeddings)
        index.add(normalized_embeddings)
        index.nprobe = runtime_parameters["nprobe"]
    else:
        raise ValueError(f"Unsupported index kind: {index_spec.kind}")

    build_seconds = float(time.perf_counter() - build_started_at)
    with tempfile.TemporaryDirectory() as temporary_dir:
        temp_index_path = Path(temporary_dir) / "index.faiss"
        faiss.write_index(index, str(temp_index_path))
        index_size_mb = float(temp_index_path.stat().st_size / (1024 ** 2))
    metadata = {"index_name": index_spec.name, "index_kind": index_spec.kind, "build_seconds": build_seconds, "index_size_mb": index_size_mb, "reference_count": int(reference_count), "embed_dim": int(embed_dim), **runtime_parameters}
    faiss.write_index(index, str(index_path))
    save_json(metadata_path, metadata)
    return index, metadata


def search_unique_track_results(index: faiss.Index, query_embeddings: np.ndarray, reference_track_ids: np.ndarray, top_k: int, oversample_factor: int) -> tuple[np.ndarray, np.ndarray]:
    """Search an index and deduplicate results to unique track identifiers."""
    normalized_queries = normalize_embeddings(query_embeddings)
    total_reference = int(reference_track_ids.shape[0])
    search_k = min(total_reference, max(top_k, top_k * oversample_factor))
    while True:
        scores, neighbor_indices = index.search(normalized_queries.astype(np.float32), search_k)
        unique_track_ids = np.full((normalized_queries.shape[0], top_k), -1, dtype=np.int64)
        unique_scores = np.full((normalized_queries.shape[0], top_k), -np.inf, dtype=np.float32)
        needs_more_neighbors = False
        for query_index in range(normalized_queries.shape[0]):
            seen_track_ids: set[int] = set()
            write_index = 0
            for score_value, reference_row_index in zip(scores[query_index], neighbor_indices[query_index]):
                if reference_row_index < 0:
                    continue
                track_id = int(reference_track_ids[reference_row_index])
                if track_id in seen_track_ids:
                    continue
                seen_track_ids.add(track_id)
                unique_track_ids[query_index, write_index] = track_id
                unique_scores[query_index, write_index] = float(score_value)
                write_index += 1
                if write_index >= top_k:
                    break
            if write_index < top_k and search_k < total_reference:
                needs_more_neighbors = True
        if not needs_more_neighbors:
            return unique_track_ids, unique_scores
        search_k = min(total_reference, search_k * 2)


def aggregate_group_rankings(query_metadata_df: pd.DataFrame, ranked_track_ids: np.ndarray, ranked_scores: np.ndarray) -> pd.DataFrame:
    """Aggregate segment-level rankings into one row per logical query group."""
    grouped_rows: list[dict[str, object]] = []
    grouped = query_metadata_df.reset_index(drop=True).groupby("query_group_id", sort=False)
    for query_group_id, group_df in grouped:
        score_buckets: dict[int, list[float]] = defaultdict(list)
        for query_row_index in group_df.index.tolist():
            for track_id, score_value in zip(ranked_track_ids[query_row_index], ranked_scores[query_row_index]):
                if track_id < 0:
                    continue
                score_buckets[int(track_id)].append(float(score_value))
        aggregated_candidates = sorted(
            [(track_id, float(np.mean(scores))) for track_id, scores in score_buckets.items()],
            key=lambda item: item[1],
            reverse=True,
        )
        true_track_id = int(group_df["track_id"].iloc[0])
        rank = next((candidate_index + 1 for candidate_index, candidate in enumerate(aggregated_candidates) if candidate[0] == true_track_id), len(aggregated_candidates) + 1)
        top_candidates = aggregated_candidates[: max(CONFIG.top_k)]
        grouped_rows.append(
            {
                "query_group_id": query_group_id,
                "true_track_id": true_track_id,
                "regime_name": str(group_df["regime_name"].iloc[0]),
                "condition_name": str(group_df["condition_name"].iloc[0]),
                "query_length_seconds": float(group_df["query_length_seconds"].iloc[0]),
                "group_size": int(group_df["group_size"].iloc[0]),
                "rank": int(rank),
                "correct_top1": bool(rank == 1),
                "top1_track_id": int(top_candidates[0][0]) if top_candidates else -1,
                "top1_score": float(top_candidates[0][1]) if top_candidates else float("nan"),
                "top_candidate_track_ids_json": json.dumps([int(candidate[0]) for candidate in top_candidates]),
                "top_candidate_scores_json": json.dumps([float(candidate[1]) for candidate in top_candidates]),
            }
        )
    return pd.DataFrame(grouped_rows)


## Smoke Tests, Full Execution, And Final Synthesis

This final execution block:
- trains enabled augmentation-policy variants when checkpoints are missing
- runs exact and approximate retrieval evaluation
- builds the long-form metrics table and the final ranked summary
- exports all required artifacts


### Smoke tests, full execution, and final exports


In [ ]:
def run_smoke_tests() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run lightweight sanity checks before the full matrix."""
    smoke_rows: list[dict[str, object]] = []
    smoke_track_ids = SPLIT_TRACK_IDS["training"][: CONFIG.smoke_test_track_count]
    candidate_specs = [
        ModelRunConfig("cnn", "baseline", 128, 1, 1e-4, 1e-4, 0.07, False),
        ModelRunConfig("hybrid_transformer", "baseline", 128, 1, 1e-4, 1e-4, 0.07, False),
        ModelRunConfig("frozen_mert", "extended_existing", 128, 1, 1e-3, 1e-4, 0.07, True),
    ]
    for run_spec in candidate_specs:
        model, input_mode, sample_rate, pair_collate_override = build_model(CONFIG, run_spec)
        model = model.to(CONFIG.device)
        collate_fn = pair_collate_override if pair_collate_override is not None else collate_pair_batch
        batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else min(CONFIG.batch_size, 4)
        dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, smoke_track_ids, int(CONFIG.segment_seconds * sample_rate), sample_rate, input_mode, CONFIG, "baseline")
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_fn)
        batch = next(iter(loader))
        batch = safe_to_device(batch, CONFIG.device)
        loss_fn = NTXentLoss(run_spec.temperature)
        optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=run_spec.learning_rate, weight_decay=run_spec.weight_decay)
        optimizer.zero_grad(set_to_none=True)
        anchor_embeddings, positive_embeddings = forward_pair_batch(model, batch, input_mode, CONFIG.device_type, CONFIG.amp_enabled)
        loss = loss_fn(anchor_embeddings, positive_embeddings)
        loss.backward()
        optimizer.step()
        smoke_rows.append({"model_name": run_spec.model_name, "input_mode": input_mode, "loss": float(loss.detach().cpu().item()), "anchor_shape": tuple(anchor_embeddings.shape), "positive_shape": tuple(positive_embeddings.shape)})
        clear_memory()

    if SPLIT_TRACK_IDS["training"]:
        visualize_preprocessing_example(CONFIG, tracks_small)
    return pd.DataFrame(smoke_rows), NOTEBOOK2_ARTIFACTS_DF


def compute_rank_metrics_from_series(ranks: pd.Series, top_k: tuple[int, ...]) -> dict[str, float]:
    """Compute Top-k accuracy and MRR from a rank series."""
    rank_values = ranks.astype(float).to_numpy()
    metrics = {f"top{k}": float(np.mean(rank_values <= k)) for k in top_k}
    metrics["mrr"] = float(np.mean(1.0 / rank_values))
    return metrics


MODEL_SMOKE_TEST_REPORT, ARTIFACT_CHECK_REPORT = run_smoke_tests()
display(MODEL_SMOKE_TEST_REPORT)
display(ARTIFACT_CHECK_REPORT)

FINAL_METRICS_LONG_DF = pd.DataFrame()
FINAL_METRICS_SUMMARY_DF = pd.DataFrame()
FAILURE_CASES_DF = pd.DataFrame()

save_dataframe(CONFIG.paths.metrics_dir / "model_smoke_test_report.csv", MODEL_SMOKE_TEST_REPORT)
save_dataframe(CONFIG.paths.metrics_dir / "artifact_check_report.csv", ARTIFACT_CHECK_REPORT)
save_json(
    CONFIG.paths.export_dir / "experiment_registry.json",
    {
        "config": CONFIG,
        "runtime": RUNTIME_INFO,
        "augmentation_policies": AUGMENTATION_POLICY_REGISTRY,
        "query_regimes": QUERY_REGIME_REGISTRY,
        "windowing_specs": WINDOWING_REGISTRY,
    },
)


# Artifact loading, training runs, full retrieval matrix, and final summary tables

In [ ]:
def build_control_artifacts() -> list[ArtifactDescriptor]:
    """Resolve the active Notebook 2 control artifacts."""
    return build_active_control_artifacts(NOTEBOOK2_ARTIFACTS_DF, CONFIG)


def load_checkpoint_into_model(descriptor: ArtifactDescriptor) -> tuple[nn.Module, InputMode, int, Callable[[list[dict[str, object]]], dict[str, object]] | None]:
    """Load a checkpointed model artifact into memory."""
    run_config = ModelRunConfig(descriptor.model_name, descriptor.augmentation_profile, descriptor.embed_dim, 1, 1e-4, 1e-4, 0.07, descriptor.freeze_backbone)
    model, input_mode, sample_rate, pair_collate_override = build_model(CONFIG, run_config)
    checkpoint_payload = torch.load(descriptor.checkpoint_path, map_location=CONFIG.device)
    if isinstance(model, FrozenMertFingerprinter):
        projection_state_dict = checkpoint_payload.get("projection_state_dict")
        if projection_state_dict is None:
            raise KeyError(f"MERT checkpoint {descriptor.checkpoint_path} is missing projection_state_dict.")
        model.projection.load_state_dict(projection_state_dict)
    else:
        model_state_dict = checkpoint_payload.get("model_state_dict")
        if model_state_dict is None:
            raise KeyError(f"Checkpoint {descriptor.checkpoint_path} is missing model_state_dict.")
        model.load_state_dict(model_state_dict)
    model = model.to(CONFIG.device)
    return model, input_mode, sample_rate, pair_collate_override


def ensure_training_artifact(run_config: ModelRunConfig) -> ArtifactDescriptor:
    """Train one augmentation-policy variant or reuse its cached checkpoint."""
    run_dir = safe_mkdir(CONFIG.paths.checkpoint_dir / run_config.run_id)
    checkpoint_path = run_dir / "checkpoint_best.pt"
    history_path = run_dir / "history.json"
    summary_path = run_dir / "training_summary.json"
    if checkpoint_path.exists() and history_path.exists() and summary_path.exists():
        return ArtifactDescriptor(run_config.run_id, "notebook3_training", run_config.model_name, run_config.augmentation_profile, run_config.embed_dim, checkpoint_path, run_config.freeze_backbone, "Cached training artifact.")

    model, input_mode, sample_rate, pair_collate_override = build_model(CONFIG, run_config)
    model = model.to(CONFIG.device)
    collate_fn = pair_collate_override if pair_collate_override is not None else collate_pair_batch
    batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else CONFIG.batch_size
    train_dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, SPLIT_TRACK_IDS["training"], int(CONFIG.segment_seconds * sample_rate), sample_rate, input_mode, CONFIG, run_config.augmentation_profile)
    val_dataset = ContrastivePairDataset(CONFIG.paths.audio_dir, SPLIT_TRACK_IDS["validation"], int(CONFIG.segment_seconds * sample_rate), sample_rate, input_mode, CONFIG, run_config.augmentation_profile)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=CONFIG.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=bool(CONFIG.num_workers > 0), worker_init_fn=seed_worker, generator=build_torch_generator(CONFIG.seed), collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=CONFIG.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=bool(CONFIG.num_workers > 0), worker_init_fn=seed_worker, generator=build_torch_generator(CONFIG.seed), collate_fn=collate_fn)
    loss_fn = NTXentLoss(run_config.temperature)
    optimizer = torch.optim.AdamW([parameter for parameter in model.parameters() if parameter.requires_grad], lr=run_config.learning_rate, weight_decay=run_config.weight_decay)
    best_val_loss = float("inf")
    best_epoch = 0
    history_rows: list[dict[str, float]] = []
    epochs_without_improvement = 0
    for epoch in range(1, run_config.epochs + 1):
        train_dataset.current_epoch = epoch - 1
        val_dataset.current_epoch = epoch - 1
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, CONFIG.device, input_mode, CONFIG)
        val_loss = validate_epoch(model, val_loader, loss_fn, CONFIG.device, input_mode, CONFIG)
        history_rows.append({"epoch": float(epoch), "train_loss": train_loss, "val_loss": val_loss})
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            if isinstance(model, FrozenMertFingerprinter):
                checkpoint_payload = {"run_config": serialize_jsonable(run_config), "projection_state_dict": model.projection.state_dict(), "backbone_name": model.model_name, "best_epoch": best_epoch, "best_val_loss": best_val_loss}
            else:
                checkpoint_payload = {"run_config": serialize_jsonable(run_config), "model_state_dict": model.state_dict(), "best_epoch": best_epoch, "best_val_loss": best_val_loss}
            torch.save(checkpoint_payload, checkpoint_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG.early_stopping_patience:
            break
    save_json(history_path, history_rows)
    save_json(summary_path, {"run_config": run_config, "best_epoch": best_epoch, "best_val_loss": best_val_loss})
    clear_memory()
    return ArtifactDescriptor(run_config.run_id, "notebook3_training", run_config.model_name, run_config.augmentation_profile, run_config.embed_dim, checkpoint_path, run_config.freeze_backbone, "Trained artifact.")


def build_metrics_long_table(detailed_results_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate detailed query-group results into the long-form metrics table."""
    if detailed_results_df.empty:
        return pd.DataFrame()
    grouping_columns = ["run_id", "source", "model_name", "augmentation_profile", "embed_dim", "windowing_name", "index_name", "index_kind", "regime_name", "condition_name", "query_length_seconds"]
    rows: list[dict[str, object]] = []
    for group_key, group_df in detailed_results_df.groupby(grouping_columns, dropna=False):
        metrics = compute_rank_metrics_from_series(group_df["rank"], CONFIG.top_k)
        row = dict(zip(grouping_columns, group_key))
        row.update(metrics)
        row["query_group_count"] = int(group_df.shape[0])
        rows.append(row)
    return pd.DataFrame(rows)


def build_final_summary_table(metrics_long_df: pd.DataFrame, faiss_results_df: pd.DataFrame) -> pd.DataFrame:
    """Build the final summary table used for ranking and recommendation logic."""
    if metrics_long_df.empty:
        return pd.DataFrame()
    configuration_columns = ["run_id", "source", "model_name", "augmentation_profile", "embed_dim", "windowing_name", "index_name", "index_kind"]
    condition_rollup = metrics_long_df.groupby(configuration_columns + ["condition_name"], dropna=False)["top1"].mean().reset_index()
    condition_pivot = condition_rollup.pivot_table(index=configuration_columns, columns="condition_name", values="top1", aggfunc="mean").reset_index()
    condition_pivot.columns = [column if isinstance(column, str) else str(column) for column in condition_pivot.columns]
    summary_df = condition_pivot.copy()
    for expected_condition in ["clean", "noise", "pitch", "time", "lowpass", "highpass", "combined_moderate", "realistic_hard"]:
        if expected_condition not in summary_df.columns:
            summary_df[expected_condition] = float("nan")
        summary_df.rename(columns={expected_condition: f"{expected_condition}_top1"}, inplace=True)
    degraded_columns = [column_name for column_name in summary_df.columns if column_name.endswith("_top1") and column_name != "clean_top1"]
    summary_df["mean_degraded_top1"] = summary_df[degraded_columns].mean(axis=1, skipna=True)
    summary_df["worst_condition_top1"] = summary_df[degraded_columns].min(axis=1, skipna=True)
    summary_df["robustness_gap"] = summary_df["clean_top1"] - summary_df["mean_degraded_top1"]
    faiss_rollup = faiss_results_df[faiss_results_df["status"] == "ok"].groupby(configuration_columns, dropna=False)[["latency_ms_per_query", "index_size_mb"]].mean().reset_index()
    summary_df = summary_df.merge(faiss_rollup, on=configuration_columns, how="left")
    summary_df.rename(columns={"latency_ms_per_query": "latency_ms"}, inplace=True)
    summary_df["mrr"] = metrics_long_df.groupby(configuration_columns, dropna=False)["mrr"].mean().reset_index()["mrr"]
    summary_df["ranking_score"] = (0.45 * summary_df["mean_degraded_top1"].fillna(0.0) + 0.20 * summary_df["worst_condition_top1"].fillna(0.0) + 0.15 * summary_df["mrr"].fillna(0.0) + 0.10 * (1.0 / (1.0 + summary_df["latency_ms"].fillna(summary_df["latency_ms"].max()))) + 0.05 * (1.0 / (1.0 + summary_df["index_size_mb"].fillna(summary_df["index_size_mb"].max()))) + 0.05 * summary_df["clean_top1"].fillna(0.0))
    return summary_df.sort_values("ranking_score", ascending=False).reset_index(drop=True)


INDEX_SPEC_REGISTRY = build_index_spec_registry(CONFIG)
ACTIVE_CONTROL_ARTIFACTS = build_control_artifacts()
TRAINED_ARTIFACTS = [ensure_training_artifact(run_config) for run_config in CONFIG.run_presets if run_config.enabled and CONFIG.run_training_ablations]
ALL_ARTIFACTS = ACTIVE_CONTROL_ARTIFACTS + TRAINED_ARTIFACTS

DETAILED_RESULT_ROWS: list[pd.DataFrame] = []
FAISS_RESULT_ROWS: list[dict[str, object]] = []
for descriptor in ALL_ARTIFACTS:
    model, input_mode, sample_rate, _ = load_checkpoint_into_model(descriptor)
    collate_fn = collate_retrieval_manifest_batch if input_mode == "spectrogram" else build_mert_retrieval_collate_fn(model.feature_extractor)  # type: ignore[arg-type]
    batch_size = CONFIG.mert_batch_size if input_mode == "mert_waveform" else CONFIG.batch_size
    for windowing_name in CONFIG.active_windowing_names:
        windowing_spec = WINDOWING_REGISTRY[windowing_name]
        if not windowing_spec.enabled:
            continue
        reference_manifest = build_reference_manifest(SPLIT_TRACK_IDS["test"], windowing_spec, CONFIG)
        reference_loader = make_retrieval_loader_v2(ReferenceManifestDataset(CONFIG.paths.audio_dir, reference_manifest, CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, CONFIG), batch_size, collate_fn, CONFIG.num_workers)
        reference_embeddings, reference_metadata_df, reference_cache = extract_embeddings_with_cache_v2(model, reference_loader, input_mode, reference_manifest, f"reference_{windowing_name}", descriptor.run_id)
        reference_track_ids = reference_metadata_df["track_id"].to_numpy(dtype=np.int64)
        for index_name in CONFIG.active_index_names:
            index_spec = INDEX_SPEC_REGISTRY[index_name]
            if not CONFIG.run_faiss_sweep and index_spec.kind != "exact_ip":
                continue
            try:
                index, index_metadata = build_or_load_faiss_index_v2(reference_embeddings, index_spec, str(reference_cache["cache_key"]))
            except Exception as exc:
                FAISS_RESULT_ROWS.append({"run_id": descriptor.run_id, "source": descriptor.source, "model_name": descriptor.model_name, "augmentation_profile": descriptor.augmentation_profile, "embed_dim": descriptor.embed_dim, "windowing_name": windowing_name, "index_name": index_name, "status": "skipped", "skip_reason": str(exc)})
                continue
            for regime_name in CONFIG.active_query_regime_names:
                regime_spec = QUERY_REGIME_REGISTRY[regime_name]
                if not regime_spec.enabled:
                    continue
                query_manifest = build_query_manifest(SPLIT_TRACK_IDS["test"], regime_spec, CONFIG)
                query_loader = make_retrieval_loader_v2(QueryManifestDataset(CONFIG.paths.audio_dir, query_manifest, CONFIG.sample_rate if input_mode == "spectrogram" else sample_rate, input_mode, CONFIG), batch_size, collate_fn, CONFIG.num_workers)
                query_embeddings, query_metadata_df, query_cache = extract_embeddings_with_cache_v2(model, query_loader, input_mode, query_manifest, f"query_{regime_name}", descriptor.run_id)
                evaluation_started_at = time.perf_counter()
                ranked_track_ids, ranked_scores = search_unique_track_results(index, query_embeddings, reference_track_ids, max(CONFIG.top_k), CONFIG.search_oversample_factor)
                detailed_results = aggregate_group_rankings(query_metadata_df, ranked_track_ids, ranked_scores)
                search_seconds = float(time.perf_counter() - evaluation_started_at)
                detailed_results["run_id"] = descriptor.run_id
                detailed_results["source"] = descriptor.source
                detailed_results["model_name"] = descriptor.model_name
                detailed_results["augmentation_profile"] = descriptor.augmentation_profile
                detailed_results["embed_dim"] = descriptor.embed_dim
                detailed_results["windowing_name"] = windowing_name
                detailed_results["index_name"] = index_name
                detailed_results["index_kind"] = index_spec.kind
                DETAILED_RESULT_ROWS.append(detailed_results)
                FAISS_RESULT_ROWS.append({"run_id": descriptor.run_id, "source": descriptor.source, "model_name": descriptor.model_name, "augmentation_profile": descriptor.augmentation_profile, "embed_dim": descriptor.embed_dim, "windowing_name": windowing_name, "index_name": index_name, "index_kind": index_spec.kind, "regime_name": regime_name, "status": "ok", "latency_ms_per_query": float(1000.0 * search_seconds / max(1, len(query_embeddings))), "query_group_count": int(detailed_results.shape[0]), "reference_embedding_path": str(reference_cache["embedding_path"]), "query_embedding_path": str(query_cache["embedding_path"]), **index_metadata})
    clear_memory()

DETAILED_RESULTS_DF = pd.concat(DETAILED_RESULT_ROWS, ignore_index=True) if DETAILED_RESULT_ROWS else pd.DataFrame()
FAISS_RESULTS_DF = pd.DataFrame(FAISS_RESULT_ROWS)
FINAL_METRICS_LONG_DF = build_metrics_long_table(DETAILED_RESULTS_DF)
FINAL_METRICS_SUMMARY_DF = build_final_summary_table(FINAL_METRICS_LONG_DF, FAISS_RESULTS_DF)
FAILURE_CASES_DF = DETAILED_RESULTS_DF[DETAILED_RESULTS_DF["correct_top1"] == False].sort_values(["condition_name", "rank"], ascending=[True, False]).head(200).reset_index(drop=True) if not DETAILED_RESULTS_DF.empty else pd.DataFrame()

save_dataframe(CONFIG.paths.export_dir / "final_metrics_long.csv", FINAL_METRICS_LONG_DF)
save_dataframe(CONFIG.paths.export_dir / "final_metrics_summary.csv", FINAL_METRICS_SUMMARY_DF)
save_dataframe(CONFIG.paths.export_dir / "failure_cases.csv", FAILURE_CASES_DF)
save_dataframe(CONFIG.paths.export_dir / "faiss_sweep_results.csv", FAISS_RESULTS_DF)
if not FINAL_METRICS_SUMMARY_DF.empty:
    save_json(CONFIG.paths.export_dir / "best_config.json", FINAL_METRICS_SUMMARY_DF.iloc[0].to_dict())
    FINAL_CONCLUSION_MARKDOWN = f"# Conclusion\n\nBest configuration: `{FINAL_METRICS_SUMMARY_DF.iloc[0]['run_id']}`\n\nMean degraded Top-1: {FINAL_METRICS_SUMMARY_DF.iloc[0]['mean_degraded_top1']:.4f}\n\nWorst-condition Top-1: {FINAL_METRICS_SUMMARY_DF.iloc[0]['worst_condition_top1']:.4f}\n"
    (CONFIG.paths.export_dir / "notebook3_conclusions.md").write_text(FINAL_CONCLUSION_MARKDOWN, encoding="utf-8")
    display(Markdown(FINAL_CONCLUSION_MARKDOWN))

display(FINAL_METRICS_SUMMARY_DF.head(10))
display(FAILURE_CASES_DF.head(20))


## Next Steps

Decision standard for Notebook 4:
- Continue only with the best configuration exported in `best_config.json`.
- If robustness improves mainly from augmentation policy changes, Notebook 4 should focus on targeted finetuning and scaling the strongest recipe rather than architecture churn.
- If realistic-hard retrieval still collapses, Notebook 4 should prioritize query-aware training and selective backbone unfreezing instead of broader augmentation stacking.

Operational note:
- Treat `notebook3_conclusions.md` as the final handoff artifact for the next notebook.
